# A.1 Detailed Analysis of Machine-Learning and Sequence-Based Models

How do manual and deepfake-aware preprocessing mathematically change Arabic text, and how are those changes associated with improved or degraded classification?

Evaluation only: no detector training, residual architecture, pseudo-perplexity, source-pair inference, or standalone linguistic study. Results are associative.

## tl;dr

Executed successfully on the available saved predictions. 90 configuration/platform/view results were reconciled to stored Macro-F1 within 1e-10.

twitter: preprocessing Macro-F1 deltas span -6.72 to +1.90 percentage points.

youtube: preprocessing Macro-F1 deltas span -1.33 to +0.43 percentage points.

Associative evidence only; one seed does not estimate training variance.

## Context and Statistical Protocol

Labels are **0 = machine-generated/fake, 1 = human/real**. “Fake” denotes provenance, not verified factual falsity. The unit is a test sample, keyed by platform and sample_idx. No reliable source-pair identifier is available. Exact-text de-leakage is not proof of source-pair independence. Twitter and YouTube remain separate throughout primary inference.

For each original→view comparison, a sample contributes one paired feature delta. Percentile bootstrap resampling therefore uses samples directly; no redundant cluster wrapper is applied. Repeated predictions across configurations are not treated as independent observations. Consensus and transition-feature profiles are descriptive, with no pooled significance tests.

Feature inference: Wilcoxon signed-rank (zero differences excluded), paired rank-biserial effect, median paired change with 95% percentile CI, and secondary original-IQR-standardized median shift. BH correction is within each platform × true class × preprocessing comparison across features. Undefined tests/zero-IQR shifts are reported, never converted to evidence of no effect.

Prediction inference: fixed-configuration Macro-F1 differences, paired sample-bootstrap CIs, and two-sided exact McNemar tests of correctness (not of Macro-F1). Precision/recall are macro-averaged. FP/FN are explicitly defined with fake as the positive class: Fake→Real is a missed fake; Real→Fake is a false alarm.

Requires Python with numpy, pandas, scipy, scikit-learn, matplotlib, openpyxl, nbformat, emoji, transformers, sentencepiece, and Pillow. Run from the project root. Each notebook is independent of the other evaluation notebooks; helper source is embedded from the new evaluation_support scripts. Platform training notebooks are read only for exact preprocessing definitions.

### Execution Parameters

**Purpose:** Set deterministic evaluation settings.

**Inputs:** Local project root and existing artifacts.

**Outputs:** Paths and bootstrap count.

**Statistical method:** Seed 42; 1,000 percentile bootstrap resamples.

**Table/figure decision:** No chart: execution settings are recorded as a table.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT=Path.cwd().resolve()
assert (ROOT/'Dataset').is_dir() and (ROOT/'Output').is_dir(), 'Run from the project root'
KIND='ml_dl'
N_BOOT=1000
OUTPUT_DIR=ROOT/'Output'/('evaluation_'+KIND)
tables={}
checks=[]

### Source-faithful data, preprocessing, and tokenizer helpers

**Purpose:** Define reusable evaluation-only functions.

**Inputs:** New evaluation_support/data.py; original scripts remain untouched.

**Outputs:** Functions used by the following named sections.

**Statistical method:** Definitions only; statistical methods are documented in each analysis section.

**Table/figure decision:** No figure or table until functions are called.

In [ ]:
"""Read-only source-artifact contracts; embedded verbatim in evaluation notebooks.

Preprocessing is extracted by function/dependency identity from the two training
notebooks, never imported from another evaluation notebook and never fitted on test.
Token measures port the former A.2 implementation, correcting special-token and
empty-text denominators and reconstructing deleted multilingual tokenizers.
"""
from pathlib import Path
import ast
import hashlib
import json
import re
import string
import unicodedata
import warnings
import numpy as np
import pandas as pd

EVAL_PLATFORMS = ("twitter", "youtube")
EVAL_VIEWS = ("original", "manual", "deepfake_aware")
EVAL_MODEL_IDS = {
    "mBERT": "google-bert/bert-base-multilingual-cased",
    "XLM-R": "FacebookAI/xlm-roberta-base",
    "AraBERTv2": "aubmindlab/bert-base-arabertv2",
    "MARBERTv2": "UBC-NLP/MARBERTv2",
    "CAMeLBERT-DA": "CAMeL-Lab/bert-base-arabic-camelbert-da",
}
EVAL_ARABIC_NAMES = {"arabertv2": "AraBERTv2", "marbertv2": "MARBERTv2", "camelbert": "CAMeLBERT-DA"}
EVAL_MODEL_NAMES = {"linearsvc": "LinearSVC", "logisticregression": "LogisticRegression", "randomforest": "RandomForest", "lstm": "LSTM", "bilstm": "Bi-LSTM"}
EVAL_EMBED_NAMES = {"tfidf": "TF-IDF", "word2vec_cbow": "Word2Vec-CBOW", "fasttext": "FastText", "random": "Random"}


def _require(checks, name, condition, detail):
    checks.append({"check": name, "passed": bool(condition), "detail": str(detail)})
    if not condition:
        raise ValueError(f"Evidence contract failed: {name}: {detail}")


def _read_csv_evidence(path):
    return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, low_memory=False)


def _normalize_eval_metadata(frame):
    out = frame.copy()
    if "topic" not in out:
        out["topic"] = out.get("field", out.get("sector", "Unassigned"))
    dialects = {
        "msa": "MSA", "modern standard arabic": "MSA", "shami": "Levantine",
        "levantine": "Levantine", "levantine arabic": "Levantine",
        "sudanese": "Sudanese", "sudanese arabic": "Sudanese", "egyptian": "Egyptian",
        "gulf": "Gulf", "iraqi": "Iraqi", "saudi": "Saudi", "kuwaiti": "Kuwaiti",
        "ads": "Unassigned", "": "Unassigned", "unknown": "Unassigned",
        "nan": "Unassigned", "none": "Unassigned",
    }
    raw = out.get("dialect", pd.Series("", index=out.index)).fillna("").astype(str).str.strip()
    out["dialect"] = raw.map(lambda x: dialects.get(x.casefold(), x))
    out["topic"] = out["topic"].fillna("").astype(str).str.strip().replace("", "Unassigned").str.title()
    subtype = out.get("deception_type", pd.Series("", index=out.index)).fillna("").astype(str).str.strip().str.lower()
    out["deception_type"] = subtype.mask(subtype.isin(["", "nan", "none"]), "Missing subtype")
    out.loc[out.y_true.eq(1), "deception_type"] = "Not applicable (real)"
    return out


def _training_preprocessors(root, platform, train, checks):
    """Execute only the transitive AST dependencies of the original functions.

    Function source is copied verbatim with ast.get_source_segment. No notebook
    cells that train, tokenize, plot, or write files are executed.
    """
    import emoji
    filename = f"Arabic_Deepfake_Detection_{'Twitter' if platform == 'twitter' else 'Youtube'}.ipynb"
    path = Path(root) / filename
    book = json.loads(path.read_text(encoding="utf-8"))
    definitions, ordinal = {}, 0
    label_maps = []
    for cell in book["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            ordinal += 1
            names = []
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                names = [node.name]
            elif isinstance(node, ast.Assign):
                names = [n.id for target in node.targets for n in ast.walk(target) if isinstance(n, ast.Name)]
            elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
                names = [node.target.id]
            if "LABEL_NAME_MAP" in names:
                try:
                    label_maps.append(ast.literal_eval(node.value))
                except (ValueError, TypeError):
                    pass
            for name in names:
                # Lower-case assignments in later modelling cells (e.g. text,
                # pattern, frame) are local identifiers inside these functions,
                # not preprocessing globals. Only source functions and constants
                # are eligible; the sole training frame is explicitly supplied.
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) or name.isupper():
                    definitions.setdefault(name, (ordinal, node, ast.get_source_segment(source, node)))
    _require(checks, f"{platform}_label_semantics_source", any(m.get(0, "").lower() == "fake" and m.get(1, "").lower() == "real" for m in label_maps), "Training LABEL_NAME_MAP: 0=Fake; 1=Real")
    needed = [f"preprocess_manual_{platform}", f"preprocess_deepfake_aware_{platform}", "get_required_special_tokens"]
    chosen = {}
    supplied = {"re", "pd", "np", "string", "unicodedata", "emoji", f"{platform}_train"}
    while needed:
        name = needed.pop()
        if name in chosen or name in supplied or name not in definitions:
            continue
        entry = definitions[name]
        chosen[name] = entry
        needed.extend(n.id for n in ast.walk(entry[1]) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load))
    code = "\n\n".join(entry[2] for _, entry in sorted({v[0]: v for v in chosen.values()}.items()))
    env = {"pd": pd, "np": np, "re": re, "string": string, "unicodedata": unicodedata, "emoji": emoji, f"{platform}_train": train}
    exec(compile(code, str(path) + "::preprocessing_dependencies", "exec"), env)
    _require(checks, f"{platform}_preprocessing_source_identity", all(name in chosen for name in [f"preprocess_manual_{platform}", f"preprocess_deepfake_aware_{platform}"]), f"Verbatim AST source; SHA256={hashlib.sha256(code.encode()).hexdigest()}; odd characters fitted on {len(train)} training rows only")
    funcs = {"original": lambda text: str(text), "manual": env[f"preprocess_manual_{platform}"], "deepfake_aware": env[f"preprocess_deepfake_aware_{platform}"]}
    return funcs, env["get_required_special_tokens"]


def _load_eval_samples(root, checks):
    parts, preprocessors, token_rules = [], {}, {}
    for platform in EVAL_PLATFORMS:
        data_root = Path(root) / "Dataset" / ("Twitter_data/Twitter_data/splits_rebuilt_v4" if platform == "twitter" else "Youtube_data/Youtube_data/splits_rebuilt")
        splits = {s: _read_csv_evidence(data_root / f"{s}_with_meta.csv") for s in ["train", "val", "test"]}
        raw_missing_subtypes = {s: int((f.label.eq(0) & f.deception_type.astype(str).str.strip().eq("")).sum()) for s, f in splits.items()}
        _require(checks, f"{platform}_raw_missing_subtype_inventory", sum(raw_missing_subtypes.values()) == (2 if platform == "youtube" else 0), f"Raw corpus split counts={raw_missing_subtypes}; report test denominator separately")
        for split, frame in splits.items():
            frame["label"] = pd.to_numeric(frame.label, errors="raise").astype(int)
            _require(checks, f"{platform}_{split}_binary_labels", set(frame.label) == {0, 1}, str(frame.label.value_counts().sort_index().to_dict()))
            plain = _read_csv_evidence(data_root / f"{split}.csv")
            _require(checks, f"{platform}_{split}_metadata_source_alignment", len(plain) == len(frame) and plain.text.astype(str).equals(frame.text.astype(str)) and np.array_equal(pd.to_numeric(plain.label).to_numpy(int), frame.label.to_numpy(int)), "Metadata-bearing split reproduces plain training-notebook split row-for-row")
        test_text = set(splits["test"].text.astype(str))
        for split in ["train", "val"]:
            frame = splits[split]
            duplicate = frame.text.astype(str).isin(test_text)
            expected = {"train": 3944, "val": 563}[split] if platform == "youtube" else 0
            _require(checks, f"{platform}_{split}_test_duplicates", int(duplicate.sum()) == expected, f"Exact raw-text matches removed={int(duplicate.sum())}; expected={expected}")
            splits[split] = frame.loc[~duplicate].reset_index(drop=True)
        expected_sizes = {"train": 15846, "val": 2265, "test": 5654} if platform == "youtube" else {"train": 19800, "val": 5657, "test": 2829}
        _require(checks, f"{platform}_deleaked_split_sizes", {s: len(f) for s, f in splits.items()} == expected_sizes, expected_sizes)
        funcs, rule = _training_preprocessors(root, platform, splits["train"], checks)
        preprocessors[platform], token_rules[platform] = funcs, rule
        original = splits["test"].rename(columns={"label": "y_true", "text": "original_text"}).reset_index(drop=True)
        original["sample_idx"] = np.arange(len(original), dtype=int)
        original["platform"] = platform
        original = _normalize_eval_metadata(original)
        missing = int((original.y_true.eq(0) & original.deception_type.eq("Missing subtype")).sum())
        _require(checks, f"{platform}_missing_test_subtypes_preserved", missing == raw_missing_subtypes["test"], f"Missing generated subtype is explicit; test support={missing}; raw corpus total={sum(raw_missing_subtypes.values())}")
        for view in EVAL_VIEWS:
            frame = original.copy()
            frame["preprocessing"] = view
            frame["processed_text"] = frame.original_text.map(funcs[view])
            parts.append(frame[["platform", "sample_idx", "preprocessing", "y_true", "original_text", "processed_text", "dialect", "topic", "deception_type"]])
    probe = "أَلف ى ؤ @user #خبر https://example.com English 123"
    twitter = preprocessors["twitter"]["deepfake_aware"](probe)
    youtube = preprocessors["youtube"]["deepfake_aware"](probe)
    divergence = all(t in twitter for t in ["<URL>", "<MENTION>", "<HASHTAG>", "<NUMBER>", "<LATIN>"]) and all(t not in youtube for t in ["<URL>", "<MENTION>", "<HASHTAG>", "<LATIN>"]) and "<NUMBER>" in youtube and "َ" in twitter and "َ" not in youtube and "أ" in youtube and "أ" not in twitter
    _require(checks, "platform_specific_deepfake_preprocessing_diverges", divergence, f"Twitter={twitter}; YouTube={youtube}; source functions remain separate")
    return pd.concat(parts, ignore_index=True), token_rules


def _canonical_legacy(frame):
    frame = frame.copy()
    frame["model"] = frame.model_name.str.lower().map(EVAL_MODEL_NAMES | EVAL_ARABIC_NAMES)
    frame["embedding"] = frame.embedding_name.map(EVAL_EMBED_NAMES).fillna("Contextual")
    frame["configuration"] = np.where(frame.model_family.eq("Transformer"), frame.model, frame.model + " | " + frame.embedding)
    return frame


def load_evidence(root, kind):
    """Return aligned predictions, sample views, stored metrics, run roster, checks."""
    if kind not in {"ml_dl", "transformers"}:
        raise ValueError("kind must be ml_dl or transformers")
    root, checks = Path(root).resolve(), []
    samples, token_rules = _load_eval_samples(root, checks)
    parts, metrics, inventory_rows = [], [], []
    for platform in EVAL_PLATFORMS:
        directory = root / "Output" / platform / "tables"
        pred = _canonical_legacy(_read_csv_evidence(directory / "in_domain_predictions_long.csv"))
        mask = pred.model_family.eq("Transformer") if kind == "transformers" else pred.model_family.isin(["ML", "Sequence"])
        pred = pred.loc[mask].copy()
        _require(checks, f"{platform}_legacy_in_domain_only", pred.evaluation_scope.eq("in_domain").all(), "No transfer/test-driven model selection or fitting")
        pred["prediction_source"] = str((directory / "in_domain_predictions_long.csv").relative_to(root))
        parts.append(pred[["platform", "sample_idx", "configuration", "model_family", "model", "embedding", "preprocessing", "y_true", "y_pred", "text", "prediction_source"]])
        met = _canonical_legacy(_read_csv_evidence(directory / "all_models_macro_f1_ci.csv"))
        met = met.loc[met.model_family.eq("Transformer") if kind == "transformers" else met.model_family.isin(["ML", "Sequence"])].copy()
        metrics.append(met[["platform", "configuration", "preprocessing", "macro_f1"]])
        for row in pred[["configuration", "model_family", "model", "embedding", "preprocessing"]].drop_duplicates().to_dict("records"):
            is_transformer = row["model_family"] == "Transformer"
            inventory_rows.append(row | {"platform": platform, "model_id": EVAL_MODEL_IDS.get(row["model"], ""), "pretraining_group": "Arabic-specific" if is_transformer else "Not applicable", "seed": 42, "max_length": 128 if row["model_family"] != "ML" else None, "physical_batch_size": 64 if row["model_family"] != "ML" else None, "gradient_accumulation_steps": 1 if row["model_family"] != "ML" else None, "effective_batch_size": 64 if row["model_family"] != "ML" else None, "evaluation_scope": "in_domain", "added_special_tokens": json.dumps(token_rules[platform](row["preprocessing"])) if is_transformer else "[]", "prediction_source": str((directory / "in_domain_predictions_long.csv").relative_to(root)), "metadata_source": f"Arabic_Deepfake_Detection_{'Twitter' if platform == 'twitter' else 'Youtube'}.ipynb"})
    if kind == "transformers":
        run_paths = sorted((root / "Output" / "multilingual_transformers" / "runs").glob("*/run_metadata.json"))
        _require(checks, "multilingual_run_count", len(run_paths) == 12, f"Found {len(run_paths)} metadata files; expected 12")
        for path in run_paths:
            meta = json.loads(path.read_text(encoding="utf-8"))
            met = json.loads((path.parent / "overall_metrics.json").read_text(encoding="utf-8"))
            pred = _read_csv_evidence(path.parent / "test_predictions.csv")
            model = {"mbert": "mBERT", "xlm_roberta": "XLM-R"}[meta["model_key"]]
            platform, view = meta["platform"], meta["preprocessing"]
            _require(checks, f"{path.parent.name}_scope", met["evaluation_scope"] == "in_domain", "Multilingual artifacts are in-domain only")
            _require(checks, f"{path.parent.name}_run_metadata", pred.platform.eq(platform).all() and pred.preprocessing.eq(view).all() and pred.model_key.eq(meta["model_key"]).all(), "Prediction columns match run metadata")
            _require(checks, f"{path.parent.name}_special_tokens", set(meta["added_special_tokens"]) == set(token_rules[platform](view)), meta["added_special_tokens"])
            _require(checks, f"{path.parent.name}_probability_label_semantics", {"prob_fake", "prob_real"}.issubset(pred.columns) and np.array_equal(pred[["prob_fake", "prob_real"]].to_numpy(float).argmax(axis=1), pred.y_pred.to_numpy(int)), "argmax([prob_fake, prob_real]) reproduces y_pred: index 0=fake; 1=real")
            pred["configuration"], pred["model"], pred["model_family"], pred["embedding"] = model, model, "Transformer", "Contextual"
            pred["prediction_source"] = str((path.parent / "test_predictions.csv").relative_to(root))
            pred = pred.rename(columns={"processed_text": "artifact_processed_text"})
            parts.append(pred[["platform", "sample_idx", "configuration", "model_family", "model", "embedding", "preprocessing", "y_true", "y_pred", "text", "artifact_processed_text", "prediction_source"]])
            metrics.append(pd.DataFrame([{"platform": platform, "configuration": model, "preprocessing": view, "macro_f1": met["macro_f1"]}]))
            inventory_rows.append({"platform": platform, "configuration": model, "model": model, "model_family": "Transformer", "embedding": "Contextual", "preprocessing": view, "model_id": meta["model_id"], "pretraining_group": "Multilingual", "seed": meta["seed"], "max_length": meta["max_len"], "physical_batch_size": meta["micro_batch_size"], "gradient_accumulation_steps": meta["gradient_accumulation_steps"], "effective_batch_size": meta["effective_batch_size"], "evaluation_scope": "in_domain", "added_special_tokens": json.dumps(meta["added_special_tokens"]), "prediction_source": str((path.parent / "test_predictions.csv").relative_to(root)), "metadata_source": str(path.relative_to(root))})
    predictions, stored, inventory = pd.concat(parts, ignore_index=True), pd.concat(metrics, ignore_index=True), pd.DataFrame(inventory_rows)
    keys = ["platform", "configuration", "preprocessing"]
    _require(checks, "binary_prediction_labels", set(predictions.y_true) == {0, 1} and set(predictions.y_pred) <= {0, 1}, "All prediction artifacts binary; source labels validated separately")
    _require(checks, "unique_prediction_keys", not predictions.duplicated(keys + ["sample_idx"]).any(), "No silently aggregated duplicate sample keys")
    _require(checks, "stored_metric_key_coverage", not stored.duplicated(keys).any() and len(stored) == (90 if kind == "ml_dl" else 30), f"Stored metric rows={len(stored)}")
    _require(checks, "result_coverage", predictions[keys].drop_duplicates().shape[0] == (90 if kind == "ml_dl" else 30), f"Observed results={predictions[keys].drop_duplicates().shape[0]}")
    _require(checks, "all_views_and_platforms", set(predictions.platform) == set(EVAL_PLATFORMS) and set(predictions.preprocessing) == set(EVAL_VIEWS), "Both platforms; all three views")
    expected_models = 15 if kind == "ml_dl" else 5
    expected_roster = set(EVAL_MODEL_IDS) if kind == "transformers" else {
        f"{model} | {embedding}" for model in ["LinearSVC", "LogisticRegression", "RandomForest"] for embedding in ["TF-IDF", "Word2Vec-CBOW", "FastText"]
    } | {
        f"{model} | {embedding}" for model in ["LSTM", "Bi-LSTM"] for embedding in ["Random", "Word2Vec-CBOW", "FastText"]
    }
    for (platform, view), group in predictions.groupby(["platform", "preprocessing"]):
        _require(checks, f"{platform}_{view}_configuration_coverage", group.configuration.nunique() == expected_models and set(group.configuration) == expected_roster, f"Exact expected roster; configurations={group.configuration.nunique()}")
    merged = predictions.merge(samples.rename(columns={"y_true": "source_y_true"}), on=["platform", "sample_idx", "preprocessing"], how="left", validate="many_to_one", indicator=True)
    _require(checks, "source_sample_metadata_alignment", merged._merge.eq("both").all() and merged.y_true.eq(merged.source_y_true).all() and merged.text.eq(merged.original_text).all(), "Every sample_idx, label and raw text equals the untouched disk test row")
    if "artifact_processed_text" in merged:
        has = merged.artifact_processed_text.notna()
        _require(checks, "multilingual_processed_text_reproduction", merged.loc[has, "artifact_processed_text"].eq(merged.loc[has, "processed_text"]).all(), f"Exact platform-source preprocessing checked on {int(has.sum())} saved multilingual rows")
    for (platform, configuration, view), group in merged.groupby(keys):
        expected = 2829 if platform == "twitter" else 5654
        _require(checks, f"prediction_support_{platform}_{configuration}_{view}", len(group) == expected and np.array_equal(np.sort(group.sample_idx.to_numpy(int)), np.arange(expected)), f"Explicit denominator={expected}")
    merged = merged.drop(columns=["source_y_true", "_merge", "text", "artifact_processed_text"], errors="ignore")
    merged["correct"] = merged.y_true.eq(merged.y_pred)
    merged["error"] = ~merged.correct
    _require(checks, "no_pair_level_analysis", True, "Only (platform, sample_idx) identifies a sample; no real/fake source-pair provenance exists")
    return merged, samples, stored, inventory, checks


def _token_measurement(tokenizer, text, max_length=128):
    """Actual content IDs include added typed tokens and UNK, not CLS/SEP/BOS/EOS.

    Tokenize without boundary tokens, so UNK and typed markers are not wrongly
    removed merely because the tokenizer lists them in all_special_ids.
    """
    ids = tokenizer(text, add_special_tokens=False, truncation=False, padding=False)["input_ids"]
    content, words = len(ids), len(text.split())
    boundary_count = tokenizer.num_special_tokens_to_add(pair=False)
    retained = max(0, max_length - boundary_count)
    return {"word_count": words, "subword_count": content, "fertility": content / max(1, words), "unknown_rate": sum(i == tokenizer.unk_token_id for i in ids) / max(1, content), "truncated": int(content + boundary_count > max_length), "retained_ratio": min(1.0, retained / content) if content else 1.0, "empty_text": int(not text.strip())}


def tokenization_evidence(root, samples, inventory, n_boot=1000):
    """Reconstruct all tokenizer/view variants; no model weights or training.

    Returns per-sample measurements, paired mean-change CIs, tokenizer manifest,
    checks. Missing tokenizer access raises a clear blocker, never fake zeros.
    """
    from transformers import AutoTokenizer
    rows, effects, manifest, checks = [], [], [], []
    for record in inventory.sort_values(["model", "platform", "preprocessing"]).to_dict("records"):
        if record["model_family"] != "Transformer":
            continue
        model, platform, view = record["model"], record["platform"], record["preprocessing"]
        model_id = record["model_id"]
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, local_files_only=True)
            origin = "Hugging Face local cache"
        except (OSError, ValueError):
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
                origin = "Hugging Face base tokenizer download"
            except Exception as exc:
                raise RuntimeError(f"Tokenizer unavailable for {model_id}. Cache/download the tokenizer files (no weights), then rerun A.2.4. Original error: {exc}") from exc
        specials = json.loads(record["added_special_tokens"])
        if specials:
            tokenizer.add_special_tokens({"additional_special_tokens": specials})
        _require(checks, f"tokenizer_specials_{platform}_{model}_{view}", all(len(tokenizer(t, add_special_tokens=False)["input_ids"]) == 1 for t in specials), f"Special tokens reconstructed from {record['metadata_source']}: {specials}")
        frame = samples.loc[samples.platform.eq(platform) & samples.preprocessing.eq(view)]
        # Suppress tokenizer advisory about untruncated diagnostic lengths only.
        tokenizer.model_max_length = 10**9
        for rec in frame[["sample_idx", "y_true", "processed_text"]].to_dict("records"):
            rows.append({"platform": platform, "sample_idx": rec["sample_idx"], "y_true": rec["y_true"], "model": model, "configuration": model, "pretraining_group": record["pretraining_group"], "preprocessing": view} | _token_measurement(tokenizer, rec["processed_text"], int(record["max_length"])))
        manifest.append({"platform": platform, "model": model, "preprocessing": view, "model_id": model_id, "tokenizer_class": type(tokenizer).__name__, "source": origin, "added_special_tokens": json.dumps(specials), "vocabulary_size": len(tokenizer), "max_length": int(record["max_length"]), "samples": len(frame), "status": "measured"})
    measurements = pd.DataFrame(rows)
    for (platform, model, label), group in measurements.groupby(["platform", "model", "y_true"]):
        original = group[group.preprocessing.eq("original")].set_index("sample_idx")
        for view in ["manual", "deepfake_aware"]:
            changed = group[group.preprocessing.eq(view)].set_index("sample_idx").reindex(original.index)
            for metric in ["fertility", "subword_count", "unknown_rate", "truncated", "retained_ratio"]:
                delta = changed[metric].to_numpy(float) - original[metric].to_numpy(float)
                rng = np.random.default_rng(42)
                boot = np.empty(n_boot)
                for start in range(0, n_boot, 100):
                    take = min(100, n_boot - start)
                    boot[start:start+take] = delta[rng.integers(0, len(delta), size=(take, len(delta)))].mean(axis=1)
                lo, hi = np.quantile(boot, [.025, .975])
                effects.append({"platform": platform, "model": model, "configuration": model, "true_class": label, "comparison": view, "metric": metric, "n": len(delta), "mean_original": float(original[metric].mean()), "mean_view": float(changed[metric].mean()), "mean_delta": float(delta.mean()), "ci_low": float(lo), "ci_high": float(hi), "resampling_unit": "sample; one paired delta per sample", "n_bootstrap": n_boot})
    _require(checks, "tokenization_coverage", measurements.groupby(["platform", "model", "preprocessing"]).ngroups == 30, "5 tokenizers x 2 platforms x 3 views; no detector embeddings or confidence compared")
    return measurements, pd.DataFrame(effects), pd.DataFrame(manifest), checks


### Ported mathematical measurements and paired inference

**Purpose:** Define reusable evaluation-only functions.

**Inputs:** New evaluation_support/patterns.py; original scripts remain untouched.

**Outputs:** Functions used by the following named sections.

**Statistical method:** Definitions only; statistical methods are documented in each analysis section.

**Table/figure decision:** No figure or table until functions are called.

In [ ]:
"""Self-contained preprocessing measurements and paired, sample-level inference.

Ported by function identity from evaluation_deepfake_patterns.ipynb. The legacy
33-feature definitions are retained, except empty repeated-sentence rate is
corrected from 1 to 0. No masked-LM, real/fake separability, or model fitting.
This source is embedded into both evaluation notebooks by the notebook builder.
"""

from collections import Counter
import hashlib
import math
import re
import warnings
import zlib

import numpy as np
import pandas as pd
from scipy.stats import rankdata, wilcoxon


ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIAC_RE = re.compile(r"[\u064b-\u065f\u0670]")
PATTERN_SEED = 42
PATTERN_VIEWS = ("original", "manual", "deepfake_aware")


def entropy(items):
    """Empirical Shannon entropy in bits; an empty collection has entropy 0."""
    counts = Counter(items)
    n = sum(counts.values())
    return -sum((value / n) * math.log2(value / n) for value in counts.values()) if n else 0.0


def ngram_stats(tokens, n):
    """Duplicate-occurrence share and unique n-gram share; both 0 if absent."""
    grams = list(zip(*[tokens[i:] for i in range(n)])) if len(tokens) >= n else []
    if not grams:
        return 0.0, 0.0
    unique = len(set(grams))
    return (len(grams) - unique) / len(grams), unique / len(grams)


def text_features(s):
    """Measure the exact supplied view; do not normalize or tokenize it again."""
    s = str(s)
    tokens = s.split()
    sentences = [x.strip() for x in re.split(r"[.!?؟؛\n]+", s) if x.strip()]
    lengths = [len(x.split()) for x in sentences] or [len(tokens)]
    counts = Counter(tokens)
    types, n = len(counts), len(tokens)
    chars = list(s)
    printable = max(1, sum(not c.isspace() for c in s))
    r1, u1 = ngram_stats(tokens, 1)
    r2, u2 = ngram_stats(tokens, 2)
    r3, u3 = ngram_stats(tokens, 3)
    runs = [len(m.group(0)) for m in re.finditer(r"(.)\1*", s)] or [0]
    compressed = len(zlib.compress(s.encode("utf-8"))) if s else 0
    raw_bytes = max(1, len(s.encode("utf-8")))
    word_entropy, character_entropy = entropy(tokens), entropy(chars)
    return {
        "characters": len(s),
        "words": n,
        "sentences": len(sentences),
        "mean_sentence_length": np.mean(lengths),
        "sentence_length_sd": np.std(lengths),
        "type_token_ratio": types / max(1, n),
        "corrected_ttr": types / math.sqrt(max(1, 2 * n)),
        "hapax_ratio": sum(v == 1 for v in counts.values()) / max(1, types),
        "vocabulary_richness": math.log1p(types) / math.log1p(max(1, n)),
        "repeated_unigram_rate": r1,
        "repeated_bigram_rate": r2,
        "repeated_trigram_rate": r3,
        "unique_unigram_ratio": u1,
        "unique_bigram_ratio": u2,
        "unique_trigram_ratio": u3,
        "max_character_run": max(runs),
        "repeated_sentence_rate": 1 - len(set(sentences)) / len(sentences) if sentences else 0.0,
        "character_entropy": character_entropy,
        "word_entropy": word_entropy,
        "normalized_character_entropy": character_entropy / math.log2(max(2, len(set(chars)))),
        "normalized_word_entropy": word_entropy / math.log2(max(2, types)),
        "word_entropy_rate": word_entropy / max(1, n),
        "sentence_burstiness": np.std(lengths) / (np.mean(lengths) + 1e-9),
        "compression_ratio": compressed / raw_bytes,
        "normalized_compressed_length": compressed / max(1, n),
        "arabic_ratio": sum(bool(ARABIC_RE.match(c)) for c in s) / printable,
        "latin_ratio": sum(bool(LATIN_RE.match(c)) for c in s) / printable,
        "mixed_script": int(bool(ARABIC_RE.search(s) and LATIN_RE.search(s))),
        "digit_ratio": sum(c.isdigit() for c in s) / printable,
        "diacritic_ratio": len(DIAC_RE.findall(s)) / printable,
        "punctuation_ratio": sum((not c.isalnum()) and (not c.isspace()) for c in s) / printable,
        "emoji_ratio": sum(ord(c) > 0x1F000 for c in s) / printable,
        "elongation_rate": sum(max(0, len(m.group(0)) - 2) for m in re.finditer(r"(.)\1{2,}", s)) / printable,
    }


def feature_dictionary():
    """Auditable formulas, including limitations of the inherited proxies."""
    definitions = [
        ("characters", "Length/structure", "len(text), including whitespace", "characters", "0"),
        ("words", "Length/structure", "N = number of whitespace-separated tokens", "tokens", "0"),
        ("sentences", "Length/structure", "S = nonempty strips after split on [.!?؟؛\\n]+", "segments", "0"),
        ("mean_sentence_length", "Length/structure", "mean segment token count; if no segments, use [N]", "tokens/segment", "0"),
        ("sentence_length_sd", "Length/structure", "population SD of segment token counts (ddof=0)", "tokens", "0"),
        ("type_token_ratio", "Lexical diversity", "V/max(1,N); V = distinct whitespace-token types", "ratio", "0"),
        ("corrected_ttr", "Lexical diversity", "V/sqrt(max(1,2N))", "index", "0"),
        ("hapax_ratio", "Lexical diversity", "number of types with frequency 1/max(1,V)", "ratio of types", "0"),
        ("vocabulary_richness", "Lexical diversity", "ln(1+V)/ln(1+max(1,N))", "index", "0"),
        ("repeated_unigram_rate", "Repetition", "(G1-U1)/G1; Gn = n-gram occurrences, Un = distinct n-grams", "ratio", "0 if G1=0"),
        ("repeated_bigram_rate", "Repetition", "(G2-U2)/G2", "ratio", "0 if G2=0"),
        ("repeated_trigram_rate", "Repetition", "(G3-U3)/G3", "ratio", "0 if G3=0"),
        ("unique_unigram_ratio", "Repetition", "U1/G1", "ratio", "0 if G1=0"),
        ("unique_bigram_ratio", "Repetition", "U2/G2", "ratio", "0 if G2=0"),
        ("unique_trigram_ratio", "Repetition", "U3/G3", "ratio", "0 if G3=0"),
        ("max_character_run", "Repetition", "max length matched by (.)\\1*; newline not matched", "characters", "0"),
        ("repeated_sentence_rate", "Repetition", "1 - number of distinct segments/S", "ratio", "0 if S=0 (corrected legacy empty-text artifact)"),
        ("character_entropy", "Information", "H(C) = -sum p(c) log2 p(c); whitespace included", "bits", "0"),
        ("word_entropy", "Information", "H(W) = -sum p(w) log2 p(w)", "bits", "0"),
        ("normalized_character_entropy", "Information", "H(C)/log2(max(2,distinct characters))", "ratio", "0"),
        ("normalized_word_entropy", "Information", "H(W)/log2(max(2,V))", "ratio", "0"),
        ("word_entropy_rate", "Information", "H(W)/max(1,N); length-normalized entropy, NOT conditional entropy rate", "bits/token index", "0"),
        ("sentence_burstiness", "Structure dispersion", "population SD segment length/(mean segment length + 1e-9)", "coefficient of variation", "0"),
        ("compression_ratio", "Compressibility", "zlib-compressed UTF-8 byte length/max(1,UTF-8 byte length); includes zlib overhead", "bytes/byte", "0"),
        ("normalized_compressed_length", "Compressibility", "zlib-compressed UTF-8 byte length/max(1,N)", "bytes/token", "0"),
        ("arabic_ratio", "Orthography", "count U+0600..U+06FF/P; P=max(1,nonwhitespace characters), includes Arabic-block punctuation/marks", "ratio", "0"),
        ("latin_ratio", "Orthography", "count ASCII A-Z or a-z/P", "ratio", "0"),
        ("mixed_script", "Orthography", "1 if Arabic-block and ASCII Latin characters both occur, else 0", "binary", "0"),
        ("digit_ratio", "Orthography", "count characters satisfying Python isdigit()/P", "ratio", "0"),
        ("diacritic_ratio", "Orthography", "count U+064B..U+065F or U+0670/P", "ratio", "0"),
        ("punctuation_ratio", "Orthography", "count nonalphanumeric nonwhitespace characters/P; includes symbols/diacritics, not pure Unicode punctuation", "ratio proxy", "0"),
        ("emoji_ratio", "Orthography", "count codepoints > U+1F000/P; inherited supplementary-codepoint proxy, not complete emoji segmentation", "ratio proxy", "0"),
        ("elongation_rate", "Orthography", "sum max(0,run_length-2) for (.)\\1{2,}/P; includes nonletter runs", "ratio proxy", "0"),
    ]
    result = pd.DataFrame(definitions, columns=["feature", "family", "definition", "unit", "empty_or_unavailable_handling"])
    result["measurement_scope"] = "processed_text exactly as supplied; whitespace tokens; no additional normalization"
    result["interpretation"] = "paired preprocessing description; association is not causal; no independent classifier fitting"
    return result


def bh(p):
    """Benjamini-Hochberg within the supplied family; missing tests stay missing."""
    values = np.asarray(p, dtype=float)
    finite = np.isfinite(values)
    if np.any((values[finite] < 0) | (values[finite] > 1)):
        raise ValueError("BH requires p-values between 0 and 1.")
    result = np.full(values.shape, np.nan)
    valid = values[finite]
    if not len(valid):
        return result
    order = np.argsort(valid, kind="stable")
    ranked = valid[order] * len(valid) / np.arange(1, len(valid) + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1]
    mapped = np.empty_like(adjusted)
    mapped[order] = np.minimum(1, adjusted)
    result[finite] = mapped
    return result


def _pattern_seed(*parts):
    payload = "|".join(map(str, (PATTERN_SEED,) + parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")


def paired_pattern_summary(original, changed, n_boot=1000, seed=42):
    """One paired delta per independent sample; no repeated-configuration cluster."""
    if n_boot < 1:
        raise ValueError("n_boot must be a positive integer.")
    original, changed = np.asarray(original, float), np.asarray(changed, float)
    if original.shape != changed.shape or original.ndim != 1:
        raise ValueError("Original and changed must be aligned one-dimensional arrays.")
    valid = np.isfinite(original) & np.isfinite(changed)
    baseline, difference = original[valid], changed[valid] - original[valid]
    n = len(difference)
    result = {
        "n": n, "n_missing_pairs": int((~valid).sum()), "n_nonzero": 0,
        "median_delta": np.nan, "ci_low": np.nan, "ci_high": np.nan,
        "percent_increased": np.nan, "percent_decreased": np.nan, "percent_unchanged": np.nan,
        "rank_biserial": np.nan, "original_iqr": np.nan,
        "iqr_standardized_median_shift": np.nan,
        "iqr_standardized_ci_low": np.nan, "iqr_standardized_ci_high": np.nan,
        "wilcoxon_statistic": np.nan, "p_value": np.nan,
        "test_status": "no finite pairs", "scale_status": "no finite original values",
        "bootstrap_unit": "sample; exactly one paired delta per sample; clustering not required",
        "bootstrap_method": "paired percentile bootstrap of median delta, 95% CI",
        "n_boot": int(n_boot), "random_seed": int(seed),
        "effect_definition": "(W_positive-W_negative)/(W_positive+W_negative); average absolute-delta ranks, zeros excluded",
        "secondary_effect_definition": "median paired delta / original-view IQR; undefined when original IQR is zero",
    }
    if not n:
        return result
    median = float(np.median(difference))
    rng = np.random.default_rng(seed)
    if np.all(difference == difference[0]):
        lo, hi = median, median
    else:
        bootstrap = np.empty(n_boot)
        for start in range(0, n_boot, 64):
            count = min(64, n_boot - start)
            indices = rng.integers(0, n, size=(count, n))
            bootstrap[start:start + count] = np.median(difference[indices], axis=1)
        lo, hi = np.quantile(bootstrap, [0.025, 0.975])
    nonzero = difference[difference != 0]
    ranks = rankdata(np.abs(nonzero), method="average")
    rbc = float(np.sum(np.sign(nonzero) * ranks) / ranks.sum()) if len(nonzero) else 0.0
    scale = float(np.subtract(*np.quantile(baseline, [0.75, 0.25])))
    result.update({
        "n_nonzero": len(nonzero), "median_delta": median, "ci_low": float(lo), "ci_high": float(hi),
        "percent_increased": 100 * float(np.mean(difference > 0)),
        "percent_decreased": 100 * float(np.mean(difference < 0)),
        "percent_unchanged": 100 * float(np.mean(difference == 0)),
        "rank_biserial": rbc, "original_iqr": scale,
        "scale_status": "defined" if scale > 0 else "undefined: original IQR is zero",
    })
    if scale > 0:
        result.update({"iqr_standardized_median_shift": median / scale,
                       "iqr_standardized_ci_low": float(lo) / scale, "iqr_standardized_ci_high": float(hi) / scale})
    if not len(nonzero):
        result.update({"wilcoxon_statistic": 0.0, "p_value": 1.0, "test_status": "all paired deltas zero; p=1 convention"})
    else:
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always", UserWarning)
            test = wilcoxon(difference, zero_method="wilcox", alternative="two-sided", correction=False, method="auto")
        result.update({"wilcoxon_statistic": float(test.statistic), "p_value": float(test.pvalue),
                       "test_status": "two-sided Wilcoxon signed-rank; zero_method=wilcox; method=auto" +
                       ("; " + " | ".join(str(w.message) for w in captured) if captured else "")})
    return result


def preprocessing_patterns(samples, n_boot=1000):
    """Return all-view sample features, paired changes, and the 33-feature dictionary."""
    required = ["platform", "sample_idx", "preprocessing", "original_text", "processed_text", "y_true", "dialect", "topic", "deception_type"]
    missing = set(required) - set(samples.columns)
    if missing:
        raise ValueError(f"Missing sample columns: {sorted(missing)}")
    frame = samples[required].copy()
    if frame.empty:
        raise ValueError("No samples supplied for preprocessing analysis.")
    if frame[["platform", "sample_idx", "preprocessing", "y_true"]].isna().any().any():
        raise ValueError("Sample identifiers, views, and true labels cannot be missing.")
    if not set(frame.y_true.unique()) <= {0, 1}:
        raise ValueError("Label contract is 0=fake and 1=real.")
    if set(frame.preprocessing.unique()) != set(PATTERN_VIEWS):
        raise ValueError("Exactly original, manual and deepfake_aware views are required.")
    keys = ["platform", "sample_idx"]
    if frame.duplicated(keys + ["preprocessing"]).any():
        raise ValueError("Each platform/sample/view must occur exactly once, not once per classifier.")
    if not frame.groupby(keys, sort=False).preprocessing.nunique().eq(3).all():
        raise ValueError("Every sample must have all three aligned preprocessing views.")
    for col in ["y_true", "original_text", "dialect", "topic", "deception_type"]:
        if frame.groupby(keys, sort=False)[col].nunique(dropna=False).gt(1).any():
            raise ValueError(f"{col} differs across views for an aligned sample.")
    # An empty processed string is meaningful: never replace it with original text.
    frame["processed_text"] = frame.processed_text.fillna("").astype(str)
    frame = frame.sort_values(keys + ["preprocessing"]).reset_index(drop=True)
    measurements = pd.DataFrame([text_features(text) for text in frame.processed_text])
    dictionary = feature_dictionary()
    if len(measurements.columns) != 33 or list(measurements.columns) != dictionary.feature.tolist():
        raise AssertionError("Feature definitions and measured columns must agree exactly.")
    sample_features = pd.concat([frame, measurements], axis=1)
    rows = []
    for (platform, label), group in sample_features.groupby(["platform", "y_true"], sort=True):
        for feature in measurements.columns:
            paired = group.pivot(index="sample_idx", columns="preprocessing", values=feature)
            for view in PATTERN_VIEWS[1:]:
                comparison = f"{view}_minus_original"
                seed = _pattern_seed(platform, int(label), comparison, feature)
                summary = paired_pattern_summary(paired.original.to_numpy(), paired[view].to_numpy(), n_boot, seed)
                rows.append({"platform": platform, "true_class": int(label), "true_class_name": "fake" if label == 0 else "real",
                             "comparison": comparison, "feature": feature, **summary,
                             "fdr_family": f"platform={platform}|true_class={int(label)}|comparison={comparison}"})
    changes = pd.DataFrame(rows)
    changes["q_value"] = changes.groupby("fdr_family", sort=False).p_value.transform(bh)
    changes["fdr_family_test_count"] = changes.groupby("fdr_family", sort=False).p_value.transform("count")
    changes["fdr_method"] = "Benjamini-Hochberg across continuous/indicator features within platform x true_class x comparison"
    changes["significant_fdr_005"] = changes.q_value.lt(0.05)
    changes["median_ci_excludes_zero"] = changes.ci_low.gt(0) | changes.ci_high.lt(0)
    return sample_features, changes, dictionary


def synthetic_tests_patterns():
    """Small pure checks; no file I/O or real-data analysis."""
    assert len(text_features("")) == 33 and all(v == 0 for v in text_features("").values())
    positive = paired_pattern_summary([1, 2, 3, 4], [2, 3, 4, 5], n_boot=24)
    negative = paired_pattern_summary([2, 3, 4, 5], [1, 2, 3, 4], n_boot=24)
    assert positive["median_delta"] == 1 and positive["rank_biserial"] == 1
    assert negative["median_delta"] == -1 and negative["rank_biserial"] == -1
    ties = paired_pattern_summary([0, 0, 0, 0], [1, -1, 2, 0], n_boot=24)
    assert ties["rank_biserial"] == 0.5 and ties["n_nonzero"] == 3
    assert np.isnan(ties["iqr_standardized_median_shift"])
    zero = paired_pattern_summary([1, 2], [1, 2], n_boot=24)
    assert zero["p_value"] == 1 and zero["rank_biserial"] == 0 and zero["percent_unchanged"] == 100
    empty = paired_pattern_summary([], [], n_boot=24)
    assert empty["n"] == 0 and np.isnan(empty["p_value"])
    np.testing.assert_allclose(bh([0.01, 0.04, 0.03, np.nan]), [0.03, 0.04, 0.04, np.nan], equal_nan=True)
    sample_rows = []
    for i, text in enumerate(["سلام سلام؟", "hello عربي 12", ""]):
        for view in PATTERN_VIEWS:
            sample_rows.append({"platform": "twitter", "sample_idx": i, "preprocessing": view,
                                "original_text": text, "processed_text": text if view == "original" else text.replace("hello", ""),
                                "y_true": i % 2, "dialect": "Unassigned", "topic": "Example", "deception_type": "Missing"})
    features, changes, dictionary = preprocessing_patterns(pd.DataFrame(sample_rows), n_boot=24)
    assert len(features) == 9 and len(changes) == 132 and len(dictionary) == 33
    assert changes.fdr_family.nunique() == 4 and changes.fdr_family_test_count.eq(33).all()
    assert np.allclose(changes.percent_increased + changes.percent_decreased + changes.percent_unchanged, 100)
    for family, group in changes.groupby("fdr_family"):
        np.testing.assert_allclose(group.q_value, bh(group.p_value), equal_nan=True)
    return {"check": "pattern_synthetic_tests", "passed": True,
            "detail": "33 features; empty text; positive/negative/tied/zero deltas; BH; sample alignment and stratified 33-test FDR families"}


### Prediction statistics, sample-aligned associations, and exports

**Purpose:** Define reusable evaluation-only functions.

**Inputs:** New evaluation_support/analysis.py; original scripts remain untouched.

**Outputs:** Functions used by the following named sections.

**Statistical method:** Definitions only; statistical methods are documented in each analysis section.

**Table/figure decision:** No figure or table until functions are called.

In [ ]:
"""Prediction-level analyses and auditable exports; embedded in both notebooks."""
import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import f1_score

RUN_KEYS = ['platform', 'configuration', 'model_family', 'model', 'embedding']
VIEW_KEYS = RUN_KEYS + ['preprocessing']
STATES = ['unchanged_correct', 'improved', 'degraded', 'unchanged_wrong']


def seeded_rng(*key):
    digest = hashlib.sha256('|'.join(map(str, key)).encode()).digest()
    return np.random.default_rng(42 + int.from_bytes(digest[:4], 'little'))


def count_metrics(counts):
    """Four confusion cells ordered true0/pred0, true0/pred1, true1/pred0, true1/pred1."""
    c = np.asarray(counts, dtype=float)
    a, b, d, e = np.moveaxis(c, -1, 0)
    def div(x, y):
        return np.divide(x, y, out=np.zeros_like(x, dtype=float), where=y != 0)
    return np.stack([
        div(a + e, c.sum(axis=-1)),
        (div(a, a+d) + div(e, e+b))/2,
        (div(a, a+b) + div(e, e+d))/2,
        (div(2*a, 2*a+b+d) + div(2*e, 2*e+b+d))/2,
    ], axis=-1)


def performance_evidence(predictions, stored_metrics, n_boot=1000):
    rows = []
    for key, g in predictions.groupby(VIEW_KEYS, sort=True, dropna=False):
        y, p = g.y_true.to_numpy(int), g.y_pred.to_numpy(int)
        counts = np.bincount(2*y+p, minlength=4)
        values = count_metrics(counts)
        draws = seeded_rng('performance', *key).multinomial(len(g), counts/len(g), size=n_boot)
        lo, hi = np.quantile(count_metrics(draws), [.025, .975], axis=0)
        row = dict(zip(VIEW_KEYS, key), n=len(g), fake_support=int((y==0).sum()),
                   real_support=int((y==1).sum()), bootstrap_resamples=n_boot)
        for j, name in enumerate(['accuracy','macro_precision','macro_recall','macro_f1']):
            row[name], row[name+'_ci_low'], row[name+'_ci_high'] = values[j], lo[j], hi[j]
        row['ci_low'], row['ci_high'] = lo[-1], hi[-1]
        assert abs(values[-1]-f1_score(y,p,average='macro',labels=[0,1])) < 1e-12
        rows.append(row)
    perf = pd.DataFrame(rows)
    keys = ['platform','configuration','preprocessing']
    source = stored_metrics[keys+['macro_f1']].rename(columns={'macro_f1':'stored_macro_f1'})
    assert not source.duplicated(keys).any(), 'Duplicate stored metric keys'
    perf = perf.merge(source, on=keys, validate='one_to_one', how='left')
    assert perf.stored_macro_f1.notna().all(), 'Stored metric coverage is incomplete'
    perf['reconciliation_abs_error'] = (perf.macro_f1-perf.stored_macro_f1).abs()
    assert perf.reconciliation_abs_error.max() < 1e-10, 'Stored Macro-F1 mismatch'
    return perf


def paired_predictions(predictions, n_boot=1000):
    summary, class_rows, pairs = [], [], []
    for key, g in predictions.groupby(RUN_KEYS, sort=True, dropna=False):
        original = g[g.preprocessing.eq('original')].sort_values('sample_idx')
        for view in ['manual','deepfake_aware']:
            other = g[g.preprocessing.eq(view)].sort_values('sample_idx')
            assert np.array_equal(original.sample_idx, other.sample_idx)
            assert np.array_equal(original.y_true, other.y_true)
            y = original.y_true.to_numpy(int)
            a, b = original.y_pred.to_numpy(int), other.y_pred.to_numpy(int)
            correct_a, correct_b = a==y, b==y
            state = np.select([correct_a & correct_b, ~correct_a & correct_b,
                               correct_a & ~correct_b], STATES[:3], default=STATES[3])
            joint = np.bincount(4*y+2*a+b, minlength=8)
            draws = seeded_rng('paired',*key,view).multinomial(len(y),joint/len(y),size=n_boot)
            # Exact empirical paired bootstrap, efficiently sampling the eight joint outcomes.
            base_counts = np.stack([draws[:,0]+draws[:,1], draws[:,2]+draws[:,3],
                                    draws[:,4]+draws[:,5], draws[:,6]+draws[:,7]],axis=1)
            view_counts = np.stack([draws[:,0]+draws[:,2], draws[:,1]+draws[:,3],
                                    draws[:,4]+draws[:,6], draws[:,5]+draws[:,7]],axis=1)
            boot_delta = count_metrics(view_counts)[:,-1]-count_metrics(base_counts)[:,-1]
            delta = f1_score(y,b,average='macro')-f1_score(y,a,average='macro')
            counts = {s:int((state==s).sum()) for s in STATES}
            changed = counts['improved']+counts['degraded']
            lo, hi = np.quantile(boot_delta,[.025,.975])
            row = dict(zip(RUN_KEYS,key), comparison=view, n=len(y),
                       macro_f1_delta=delta, bootstrap_mean_delta=float(boot_delta.mean()),
                       ci_low=lo, ci_high=hi, error_count_delta=counts['degraded']-counts['improved'],
                       mcnemar_p=binomtest(counts['improved'],changed,.5).pvalue if changed else 1.,
                       changed_predictions=int((a!=b).sum()), bootstrap_resamples=n_boot, **counts)
            summary.append(row)
            for label in [0,1]:
                mask=y==label; n=int(mask.sum()); c={s:int(((state==s)&mask).sum()) for s in STATES}
                discordant=c['improved']+c['degraded']
                class_rows.append(dict(zip(RUN_KEYS,key),comparison=view,true_class=label,n=n,
                    error_rate_original=float((~correct_a[mask]).mean()),
                    error_rate_view=float((~correct_b[mask]).mean()),
                    mcnemar_p=binomtest(c['improved'],discordant,.5).pvalue if discordant else 1., **c))
            pair=original.copy();pair['comparison']=view;pair['transition']=state
            pair['original_prediction']=a;pair['view_prediction']=b
            pair['view_text']=other.processed_text.to_numpy()
            pairs.append(pair.drop(columns='preprocessing'))
    return pd.DataFrame(summary),pd.DataFrame(class_rows),pd.concat(pairs,ignore_index=True)


def direction_evidence(predictions):
    rows=[]
    for key,g in predictions.groupby(VIEW_KEYS,sort=True,dropna=False):
        fake=g.y_true.eq(0);real=g.y_true.eq(1)
        f2r=int((fake & g.y_pred.eq(1)).sum());r2f=int((real & g.y_pred.eq(0)).sum())
        rows.append(dict(zip(VIEW_KEYS,key),n=len(g),fake_support=int(fake.sum()),real_support=int(real.sum()),
                         fake_to_real=f2r,real_to_fake=r2f,
                         fake_to_real_rate=f2r/fake.sum(),real_to_fake_rate=r2f/real.sum(),
                         error_rate=(f2r+r2f)/len(g),positive_class_for_named_fp_fn='fake (0)',
                         false_negative_count=f2r,false_positive_count=r2f))
    return pd.DataFrame(rows)


def metadata_evidence(predictions):
    result={}
    for field in ['deception_type','dialect','topic']:
        frame=predictions[predictions.y_true.eq(0)] if field=='deception_type' else predictions
        keys=VIEW_KEYS+['y_true',field]
        temp=frame.assign(is_error=frame.y_true.ne(frame.y_pred))
        table=temp.groupby(keys,dropna=False,sort=True).is_error.agg(['size','sum']).reset_index()
        table=table.rename(columns={'size':'support','sum':'error_count','y_true':'true_class',field:'category'})
        table['error_rate']=table.error_count/table.support
        table['metadata_field']=field
        table['low_support']=table.support.lt(30)
        expected=temp.groupby(VIEW_KEYS+['y_true'],dropna=False).size().sort_index()
        actual=table.groupby(VIEW_KEYS+['true_class'],dropna=False).support.sum().sort_index()
        assert np.array_equal(expected.to_numpy(),actual.to_numpy()), 'Category denominator loss'
        result['errors_by_'+field]=table
    return result


def agreement_evidence(predictions):
    votes=predictions.assign(is_error=predictions.y_true.ne(predictions.y_pred)).groupby(
        ['platform','sample_idx','preprocessing'],sort=True).agg(
            error_votes=('is_error','sum'),configuration_count=('configuration','nunique'),
            true_class=('y_true','first'),original_text=('original_text','first'),
            dialect=('dialect','first'),topic=('topic','first'),deception_type=('deception_type','first')).reset_index()
    votes['error_vote_rate']=votes.error_votes/votes.configuration_count
    votes['all_configurations_wrong']=votes.error_votes.eq(votes.configuration_count)
    votes=votes.sort_values(['platform','preprocessing','error_votes','sample_idx'],ascending=[True,True,False,True])
    overlaps=[]
    for (platform,view),g in predictions.groupby(['platform','preprocessing']):
        configs=sorted(g.configuration.unique())
        err={c:set(g.loc[g.configuration.eq(c)&g.y_true.ne(g.y_pred),'sample_idx']) for c in configs}
        names=g.groupby('configuration').model.first().to_dict()
        for a in configs:
            for b in configs:
                union=len(err[a]|err[b]);inter=len(err[a]&err[b])
                overlaps.append(dict(platform=platform,preprocessing=view,configuration_a=a,configuration_b=b,
                    model_a=names[a],model_b=names[b],intersection=inter,union=union,
                    jaccard=inter/union if union else np.nan,empty_union=not bool(union),n=g.sample_idx.nunique()))
    return votes,pd.DataFrame(overlaps)


def example_evidence(predictions,pairs,votes,per_group=3):
    mistakes=predictions[predictions.y_true.ne(predictions.y_pred)].copy()
    mistakes['error_direction']=np.where(mistakes.y_true.eq(0),'Fake→Real','Real→Fake')
    hard=votes[votes.error_votes.gt(0)].groupby(['platform','preprocessing'],sort=True).head(per_group).copy()
    hard['selection_rule']='highest error votes; ties by ascending sample_idx'
    sensitive=pairs[pairs.transition.isin(['improved','degraded'])].sort_values(
        ['platform','configuration','comparison','transition','sample_idx'])
    sensitive=sensitive.groupby(['platform','configuration','comparison','transition'],sort=True).head(per_group)
    sensitive=sensitive.copy();sensitive['selection_rule']='first sample_idx per configuration/comparison/transition'
    return mistakes,hard,sensitive


def transition_feature_profiles(sample_features,pairs):
    """Descriptive associations, one sample per configuration/state; no pooled pseudo-replication."""
    ids=['platform','sample_idx']
    excluded={'platform','sample_idx','preprocessing','y_true','true_class','original_text','processed_text',
              'dialect','topic','deception_type','text'}
    features=[c for c in sample_features.select_dtypes(include='number').columns if c not in excluded]
    original=sample_features[sample_features.preprocessing.eq('original')].set_index(ids)
    outputs=[]
    for comparison in ['manual','deepfake_aware']:
        other=sample_features[sample_features.preprocessing.eq(comparison)].set_index(ids)
        assert original.index.equals(other.index) or set(original.index)==set(other.index)
        delta=other[features].subtract(original[features]).reset_index()
        matched=pairs[pairs.comparison.eq(comparison)].merge(delta,on=ids,validate='many_to_one')
        group_cols=RUN_KEYS+['comparison','y_true','transition']
        for key,g in matched.groupby(group_cols,sort=True,dropna=False):
            for f in features:
                outputs.append(dict(zip(group_cols,key),feature=f,support=len(g),
                    median_feature_delta=float(g[f].median()),mean_feature_delta=float(g[f].mean()),
                    q25_feature_delta=float(g[f].quantile(.25)),q75_feature_delta=float(g[f].quantile(.75)),
                    low_support=len(g)<30))
    return pd.DataFrame(outputs).rename(columns={'y_true':'true_class'})


def export_evidence(tables,figure_manifest,checks,output_dir):
    """Exact CSVs plus a consolidated, filterable workbook. Never evaluates text as Excel formulas."""
    from openpyxl import Workbook, load_workbook
    from openpyxl.cell import WriteOnlyCell
    from openpyxl.styles import Font, PatternFill
    output_dir=Path(output_dir);table_dir=output_dir/'tables';table_dir.mkdir(parents=True,exist_ok=True)
    all_tables=dict(tables,figure_manifest=figure_manifest,validation_checks=pd.DataFrame(checks))
    for rec in figure_manifest.to_dict('records'):
        backing=Path(rec['backing_csv'])
        if not backing.is_absolute():backing=output_dir/backing
        assert backing.is_file(), f'Missing backing CSV: {backing}'
        all_tables[backing.stem]=pd.read_csv(backing,encoding='utf-8-sig')
    manifest=[];wb=Workbook(write_only=True)
    used=set()
    def sheet_name(name):
        base=re.sub(r'[\\/*?:\[\]]','_',name)[:27];candidate=base;counter=1
        while candidate in used:
            candidate=base[:26]+'_'+str(counter);counter+=1
        used.add(candidate);return candidate
    for name,frame in all_tables.items():
        assert len(frame)<1048576, f'{name} exceeds Excel sheet row limit'
        csv_path=table_dir/(name+'.csv');frame.to_csv(csv_path,index=False,encoding='utf-8-sig')
        ws=wb.create_sheet(sheet_name(name));ws.freeze_panes='A2'
        headers=[]
        for col in frame.columns:
            cell=WriteOnlyCell(ws,value=str(col));cell.data_type='s';cell.font=Font(bold=True,color='FFFFFF')
            cell.fill=PatternFill('solid',fgColor='26313B');headers.append(cell)
        ws.append(headers)
        for values in frame.itertuples(index=False,name=None):
            row=[]
            for value in values:
                if isinstance(value,np.generic):value=value.item()
                if isinstance(value,(dict,list,tuple)):value=json.dumps(value,ensure_ascii=False)
                if value is None or (not isinstance(value,str) and pd.isna(value)):value=None
                if isinstance(value,str):
                    if len(value)>32767:
                        raise ValueError(f'Excel text limit exceeded in {name}; CSV remains complete')
                    value=re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]','',value)
                    cell=WriteOnlyCell(ws,value=value);cell.data_type='s';row.append(cell)
                else:row.append(value)
            ws.append(row)
        if len(frame.columns):
            from openpyxl.utils import get_column_letter
            ws.auto_filter.ref=f'A1:{get_column_letter(len(frame.columns))}{len(frame)+1}'
        manifest.append(dict(table=name,rows=len(frame),columns=len(frame.columns),
                             csv=str(csv_path),excel_sheet=ws.title))
    manifest_frame=pd.DataFrame(manifest)
    manifest_frame.to_csv(table_dir/'table_manifest.csv',index=False,encoding='utf-8-sig')
    ws=wb.create_sheet('table_manifest');ws.append(list(manifest_frame.columns))
    for row in manifest_frame.itertuples(index=False,name=None):ws.append(list(row))
    path=output_dir/(output_dir.name+'_results.xlsx');wb.save(path)
    # Write-only worksheets omit dimensions. Reopening them with openpyxl can
    # scan every long-text row just to infer dimensions; validate package/schema
    # here instead. Small numeric/string round trips are covered by unit tests.
    import zipfile
    from xml.etree import ElementTree as ET
    with zipfile.ZipFile(path) as package:
        tree=ET.fromstring(package.read('xl/workbook.xml'))
        sheets=tree.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheets/{http://schemas.openxmlformats.org/spreadsheetml/2006/main}sheet')
        assert len(sheets)==len(manifest)+1
    return manifest_frame,path


def synthetic_tests_analysis():
    from sklearn.metrics import accuracy_score, precision_score, recall_score
    for y,p in [(np.array([0,0,1,1]),np.array([0,1,0,1])),
                (np.array([0,0,0,1]),np.array([0,0,1,1]))]:
        actual=count_metrics(np.bincount(2*y+p,minlength=4))
        expected=[accuracy_score(y,p),precision_score(y,p,average='macro',zero_division=0),
                  recall_score(y,p,average='macro'),f1_score(y,p,average='macro')]
        np.testing.assert_allclose(actual,expected,atol=1e-14)
    assert binomtest(2,2,.5).pvalue==.5
    return True


### Copied/adapted scientific figure styling and evidence gates

**Purpose:** Define reusable evaluation-only functions.

**Inputs:** New evaluation_support/figures.py; original scripts remain untouched.

**Outputs:** Functions used by the following named sections.

**Statistical method:** Definitions only; statistical methods are documented in each analysis section.

**Table/figure decision:** No figure or table until functions are called.

In [ ]:
"""Table-backed, evaluation-only figures, embedded into both notebook builds.

Chart contract: A.1 compares ML/sequence configurations and A.2 compares the
five transformers. These groups never share a figure. Original-condition
performance is descriptive; paired changes and intervals are precomputed by
the statistical code. Figure selection never changes a model or a result.

Style provenance: adapted from ``exact_model_preprocessing_figures_helpers.py``
(_blend_color, _x_positions and plot_preprocessing_exact_model_bars). The
original helper is not imported or modified; its model-selection code is not
reused. Focused bar axes are explicitly disclosed per the user specification.
"""

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, FuncFormatter


FIGURE_MODELS = ["mBERT", "XLM-R", "AraBERTv2", "MARBERTv2", "CAMeLBERT-DA"]
FIGURE_PLATFORMS = ["twitter", "youtube"]
FIGURE_VIEWS = ["original", "manual", "deepfake_aware"]
FIGURE_FAMILY_COLOURS = {"ML": "#C44E52", "Sequence": "#55A868"}
FIGURE_TRANSITION_COLOURS = ["#DDE7EF", "#AEC3D7", "#6E96B8", "#294F70"]


def _figure_slug(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def _figure_blend(colour, amount):
    rgb = np.asarray(to_rgb(colour))
    return tuple(rgb + (1.0 - rgb) * amount)


def _figure_font():
    installed = {font.name for font in fm.fontManager.ttflist}
    choices = ["Latin Modern Roman", "Latin Modern Sans", "Computer Modern Roman",
               "CMU Serif", "cmr10", "DejaVu Serif"]
    return next((name for name in choices if name in installed), "DejaVu Serif")


def _figure_style(ax, grid_axis="y"):
    ax.set_facecolor("white")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(1.05)
        ax.spines[side].set_color("#3D4A56")
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, linestyle="--", linewidth=0.8, alpha=0.25, color="#8C96A3")
    ax.tick_params(colors="#26313B")


def _figure_legend(ax, handles, columns=2, fontsize=8.0):
    ax.legend(handles=handles, frameon=False, loc="upper center",
              bbox_to_anchor=(0.5, 1.20), fontsize=fontsize, ncol=columns,
              columnspacing=1.2, handlelength=1.4)


def _figure_platform(value):
    return "Twitter/X" if value == "twitter" else "YouTube"


def _figure_comparison(value):
    return "Deepfake-aware - original" if "deepfake" in str(value) else "Manual - original"


def _figure_short_label(row):
    model = str(row["model"])
    model_map = {"LinearSVC": "LSVC", "LogisticRegression": "LR", "RandomForest": "RF",
                 "Logistic Regression": "LR", "Random Forest": "RF", "SVM": "LSVC"}
    embedding = str(row.get("embedding", ""))
    embedding_map = {"TF-IDF": "TFIDF", "tfidf": "TFIDF", "Word2Vec-CBOW": "W2V-CBOW",
                     "word2vec": "W2V-CBOW", "Random": "Random", "random": "Random"}
    return model_map.get(model, model) + "\n" + embedding_map.get(embedding, embedding)


def _figure_order(frame, kind):
    if kind == "transformers":
        key = "model"
        order = FIGURE_MODELS
    else:
        key = "configuration"
        roster = frame[["configuration", "model_family", "model", "embedding"]].drop_duplicates()
        roster = roster.assign(_family=roster.model_family.map({"ML": 0, "Sequence": 1}))
        roster = roster.sort_values(["_family", "model", "embedding", "configuration"])
        order = roster.configuration.tolist()
    return key, order


def _figure_support_reason(frame, columns=("n",)):
    if frame.empty:
        return "No eligible evidence rows; retain the table."
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce")
        if values.isna().any() or (values < 30).any():
            return f"At least one plotted category has fewer than 30 samples or missing {column}."
    return ""


def _figure_effect_reason(frame):
    reason = _figure_support_reason(frame)
    if reason:
        return reason
    if not np.isfinite(frame[["ci_low", "ci_high"]].to_numpy(dtype=float)).all():
        return "At least one confidence interval is undefined; retain the exact table."
    if ((frame.ci_low <= 0) & (frame.ci_high >= 0)).all():
        return "All effect confidence intervals cross or touch zero; the table is more informative."
    return ""


def _figure_intervals(ax, values, positions, lo, hi, colours, horizontal=True, marker="o"):
    """Preserve the actual interval even when it does not contain its estimate."""
    for value, pos, low, high, colour in zip(values, positions, lo, hi, colours):
        if not np.isfinite([value, low, high]).all() or low > high:
            raise ValueError("Invalid figure estimate or confidence interval.")
        if low <= value <= high:
            errors = np.array([[value - low], [high - value]])
            kwargs = {"xerr" if horizontal else "yerr": errors}
            ax.errorbar(value if horizontal else pos, pos if horizontal else value,
                        fmt=marker, markersize=5, color=colour, ecolor="#26313B",
                        elinewidth=0.9, capsize=2.8, capthick=0.9, zorder=4, **kwargs)
        elif horizontal:
            ax.hlines(pos, low, high, color="#26313B", linewidth=0.9, zorder=4)
            ax.plot([low, high], [pos, pos], "|", color="#26313B", markersize=5.6)
            if marker != "none":
                ax.plot(value, pos, marker=marker, color=colour, markersize=5, zorder=5)
        else:
            ax.vlines(pos, low, high, color="#26313B", linewidth=0.9, zorder=4)
            ax.plot([pos, pos], [low, high], "_", color="#26313B", markersize=5.6)


def generate_figures(kind, tables, output_dir):
    """Export specified candidates and their authoritative backing CSVs.

    ``kind`` is ``ml_dl`` or ``transformers``. Inputs must be validated evidence
    tables, not raw predictions. Empty/missing optional tables are documented
    rather than replaced by invented measurements. Figure-specific decisions
    are returned as a DataFrame for the consolidated workbook and manifest.
    """
    if kind not in {"ml_dl", "transformers"}:
        raise ValueError("kind must be ml_dl or transformers")
    output_dir = Path(output_dir)
    figure_dir, table_dir = output_dir / "figures", output_dir / "tables"
    figure_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    font = _figure_font()
    model_colours = dict(zip(FIGURE_MODELS, ["#C44E52", _figure_blend("#C44E52", 0.35),
                        "#4C72B0", _figure_blend("#4C72B0", 0.23), _figure_blend("#4C72B0", 0.46)]))

    def source(name):
        frame = tables.get(name, pd.DataFrame()).copy()
        if not frame.empty and "model_family" in frame:
            families = set(frame.model_family.astype(str))
            if kind == "ml_dl" and not families <= {"ML", "Sequence"}:
                raise ValueError(f"Non-ML/sequence data in A.1 table {name}.")
            if kind == "transformers" and families & {"ML", "Sequence"}:
                raise ValueError(f"ML/sequence data in A.2 table {name}.")
        return frame

    def colour(row):
        if kind == "transformers":
            return model_colours[row["model"]]
        return FIGURE_FAMILY_COLOURS[row["model_family"]]

    def caption_colour():
        return ("Colour represents model family: ML red; sequence green." if kind == "ml_dl" else
                "Colour represents pretraining group: multilingual red shades; Arabic-specific blue shades.")

    def record(stem, frame, caption, reason="", fig=None, decision=""):
        csv_path = table_dir / f"{stem}_figure_data.csv"
        frame.to_csv(csv_path, index=False, encoding="utf-8-sig")
        row = dict(figure_id=stem, kind=kind, status="suppressed" if reason else "exported",
                   reason=reason or decision or "Evidence and support gates passed.", caption=caption,
                   png="", pdf="", backing_csv=str(csv_path.relative_to(output_dir)), font=font,
                   layout_check="not_applicable")
        if fig is not None:
            if reason:
                plt.close(fig)
                raise ValueError("A suppressed candidate must not be rendered.")
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            bounds = fig.get_tightbbox(renderer)
            # Tight export encloses axis labels, titles, legends and count labels.
            # This detects invalid artist geometry, but is not a substitute for visual QA.
            if bounds is None or not np.isfinite(bounds.extents).all():
                plt.close(fig)
                raise ValueError(f"Nonfinite figure bounds: {stem}")
            png_path, pdf_path = figure_dir / f"{stem}.png", figure_dir / f"{stem}.pdf"
            fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.12, facecolor="white")
            fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.12, facecolor="white")
            row.update(png=str(png_path.relative_to(output_dir)), pdf=str(pdf_path.relative_to(output_dir)),
                       layout_check="finite artist extents; tight bounding-box export; visual review required")
            plt.close(fig)
        elif not reason:
            raise ValueError("Exported figure needs a figure object.")
        manifest.append(row)

    performance = source("performance")
    changes = source("prediction_changes")
    directions = source("error_directions")
    if not performance.empty:
        expected = 15 if kind == "ml_dl" else 5
        observed_families = set(performance.model_family.astype(str))
        if kind == "ml_dl" and not observed_families <= {"ML", "Sequence"}:
            raise ValueError("Transformer data must never enter A.1 figures.")
        if kind == "transformers" and set(performance.model) != set(FIGURE_MODELS):
            raise ValueError("A.2 figures require exactly the five specified transformer models.")
        counts = performance.groupby(["platform", "preprocessing"]).size()
        if len(counts) != 6 or not counts.eq(expected).all():
            raise ValueError("Performance coverage must have all six complete platform/view panels.")
        if performance.duplicated(["platform", "preprocessing", "configuration"]).any():
            raise ValueError("Duplicate configuration rows cannot be drawn as independent results.")

    settings = {"font.family": font, "font.size": 9, "axes.labelcolor": "#26313B",
                "text.color": "#26313B", "axes.titlesize": 11, "figure.facecolor": "white",
                "axes.formatter.use_mathtext": True, "pdf.fonttype": 42, "ps.fonttype": 42}
    with plt.rc_context(settings):
        if kind == "ml_dl":
            _draw_ml_performance(performance, record, caption_colour)
            _draw_prediction_forest(changes, record, colour, caption_colour)
        else:
            _draw_transformer_performance(performance, record, model_colours, caption_colour)
            _draw_tokenization(source("tokenization_effects"), record, model_colours, caption_colour)
            _draw_jaccard(source("error_jaccard"), record)
        _draw_error_directions(directions, kind, record, colour, caption_colour)
        _draw_transitions(changes, kind, record)
    return pd.DataFrame(manifest)


def _draw_ml_performance(frame, record, caption_colour):
    caption = ("Macro-F1 and 95% test-sample bootstrap intervals for all 15 ML/sequence configurations, "
               "separately by platform and preprocessing. All six panels share an explicitly zoomed, nonzero y-axis: "
               "compare values and intervals, not bar-length ratios. " + caption_colour())
    reason = _figure_support_reason(frame)
    if not reason:
        spreads = frame.groupby(["platform", "preprocessing"]).macro_f1.agg(lambda x: x.max() - x.min())
        if (spreads < 0.02).any():
            reason = "At least one required panel has model spread below 2 percentage points; retain the full exact table."
    if reason:
        record("F1_configuration_performance", frame, caption, reason)
        return
    _, order = _figure_order(frame, "ml_dl")
    roster = frame.drop_duplicates("configuration").set_index("configuration").loc[order]
    family_mean = frame.groupby(["model_family", "configuration"]).macro_f1.mean()
    shades = {}
    for family in ("ML", "Sequence"):
        ranking = family_mean.loc[family].sort_values(ascending=False, kind="stable")
        for index, config in enumerate(ranking.index):
            shades[config] = _figure_blend(FIGURE_FAMILY_COLOURS[family], 0.48 * index / max(1, len(ranking) - 1))
    fig, axes = plt.subplots(3, 2, figsize=(18, 16), layout="constrained", sharey=True)
    lower = max(0.0, np.floor((float(frame.ci_low.min()) - 0.015) / 0.02) * 0.02)
    upper = min(1.0, np.ceil((float(frame.ci_high.max()) + 0.03) / 0.02) * 0.02)
    for row_index, view in enumerate(FIGURE_VIEWS):
        for col_index, platform in enumerate(FIGURE_PLATFORMS):
            ax = axes[row_index, col_index]
            part = frame[(frame.platform == platform) & (frame.preprocessing == view)].set_index("configuration").loc[order]
            pos = np.arange(len(part), dtype=float) * 0.88
            pos[9:] += 0.34
            colours = [shades[name] for name in order]
            ax.bar(pos, part.macro_f1, width=0.80, edgecolor="#3D4A56", linewidth=1.0, zorder=3, color=colours)
            _figure_intervals(ax, part.macro_f1, pos, part.ci_low, part.ci_high, colours, horizontal=False, marker="none")
            ax.axvline((pos[8] + pos[9]) / 2, color="#D3D8DE", linestyle=":", linewidth=0.8)
            ax.set_xticks(pos, [_figure_short_label(row) for _, row in roster.iterrows()], rotation=58, ha="right", fontsize=10.5)
            ax.tick_params(axis="x", pad=2)
            ax.set_ylim(lower, upper)
            ax.set_xlim(pos.min() - 0.55, pos.max() + 0.55)
            ax.yaxis.set_major_locator(MultipleLocator(0.05))
            ax.tick_params(axis="y", labelleft=True, labelsize=11)
            ax.set_title(f"{_figure_platform(platform)} | {view.replace('_', ' ').title()}", fontsize=15)
            ax.set_ylabel("Macro-F1 (zoomed axis)", fontweight="semibold", fontsize=13)
            _figure_style(ax)
    _figure_legend(axes[0, 0], [Patch(facecolor=FIGURE_FAMILY_COLOURS[f], label=f) for f in ("ML", "Sequence")], fontsize=12)
    record("F1_configuration_performance", frame, caption, fig=fig)


def _draw_prediction_forest(frame, record, colour, caption_colour):
    caption = ("Paired Macro-F1 changes in percentage points, relative to original preprocessing. "
               "Whiskers are paired sample-bootstrap 95% intervals; zero indicates no change. " + caption_colour())
    reason = _figure_effect_reason(frame)
    if reason:
        record("F2_preprocessing_macro_f1_effects", frame, caption, reason)
        return
    _, order = _figure_order(frame, "ml_dl")
    comparisons = sorted(frame.comparison.unique(), key=lambda value: ("deepfake" in value, value))
    fig, axes = plt.subplots(2, len(comparisons), figsize=(14, 11), layout="constrained", squeeze=False)
    extent = float(np.nanmax(np.abs(frame[["ci_low", "ci_high"]].to_numpy()))) * 100 * 1.12
    for row_index, platform in enumerate(FIGURE_PLATFORMS):
        for col_index, comparison in enumerate(comparisons):
            ax = axes[row_index, col_index]
            part = frame[(frame.platform == platform) & (frame.comparison == comparison)].set_index("configuration").loc[order].reset_index()
            pos = np.arange(len(part))
            _figure_intervals(ax, part.macro_f1_delta * 100, pos, part.ci_low * 100, part.ci_high * 100,
                              [colour(row) for _, row in part.iterrows()])
            ax.set_yticks(pos, [_figure_short_label(row).replace("\n", " / ") for _, row in part.iterrows()], fontsize=8)
            ax.invert_yaxis()
            ax.axvline(0, color="#8C96A3", linestyle="--", linewidth=0.9)
            ax.set_xlim(-extent, extent)
            ax.set_xlabel("Macro-F1 change (percentage points)")
            ax.set_title(f"{_figure_platform(platform)} | {_figure_comparison(comparison)}")
            _figure_style(ax, "x")
    record("F2_preprocessing_macro_f1_effects", frame, caption, fig=fig)


def _draw_transformer_performance(frame, record, colours, caption_colour):
    original = frame[frame.preprocessing == "original"].copy() if not frame.empty else frame
    caption = ("Original-view Macro-F1 with 95% test-sample bootstrap intervals. The equally wide platform panels "
               "use independently zoomed x-axes: compare numeric intervals, not their physical widths across panels. "
               "Models are ordered multilingual first, then Arabic-specific. " + caption_colour())
    spreads = original.groupby("platform").macro_f1.agg(lambda x: x.max() - x.min()) if not original.empty else pd.Series(dtype=float)
    spread_text = "; ".join(f"{_figure_platform(p)}: {100 * s:.2f} pp" for p, s in spreads.items())
    record("F5_grouped_bars_suppressed", frame, caption,
           "Grouped transformer bars are excluded by design; small original-view spreads favour exact tables and zoomed dots. " + spread_text)
    reason = _figure_support_reason(original)
    if reason:
        record("F5_transformer_original_dot_ci", original, caption, reason)
        return
    fig, axes = plt.subplots(2, 1, figsize=(8.8, 7.5), layout="constrained")
    for ax, platform in zip(axes, FIGURE_PLATFORMS):
        part = original[original.platform == platform].set_index("model").loc[FIGURE_MODELS]
        pos = np.arange(5)
        _figure_intervals(ax, part.macro_f1, pos, part.ci_low, part.ci_high, [colours[model] for model in FIGURE_MODELS])
        ax.set_yticks(pos, FIGURE_MODELS)
        ax.invert_yaxis()
        ax.axhline(1.5, color="#D3D8DE", linestyle=":", linewidth=1.1)
        ax.set_xlim(float(part.ci_low.min()) - 0.002, float(part.ci_high.max()) + 0.002)
        ax.set_xlabel("Macro-F1 (panel-specific zoomed scale)", fontweight="semibold")
        ax.set_title(_figure_platform(platform))
        _figure_style(ax, "x")
        for tick, model in zip(ax.get_yticklabels(), FIGURE_MODELS):
            tick.set_color(colours[model])
    record("F5_transformer_original_dot_ci", original, caption, fig=fig,
           decision="Explicit F5 exception: retain zoomed dot-and-CI below 2 pp; grouped bars suppressed. " + spread_text)


def _draw_tokenization(frame, record, colours, caption_colour):
    metrics = ["fertility", "subwords", "unknown_rate", "truncation_rate", "retained_ratio"]
    aliases = {"subword_count": "subwords", "n_subwords": "subwords", "unknown_token_rate": "unknown_rate",
               "truncated": "truncation_rate", "truncation": "truncation_rate"}
    if not frame.empty:
        frame["metric"] = frame.metric.replace(aliases)
    for metric in metrics:
        metric_frame = frame[frame.metric == metric].copy() if not frame.empty else frame
        classes = sorted(metric_frame.true_class.unique(), key=str) if not metric_frame.empty else ["all"]
        for true_class in classes:
            part_class = metric_frame[metric_frame.true_class == true_class] if not metric_frame.empty else metric_frame
            class_label = {0: "fake/machine-generated (0)", 1: "real/human (1)"}.get(true_class, str(true_class))
            caption = (f"Mean paired change in {metric.replace('_', ' ')} relative to the original input, true class {class_label}. "
                       "Each point is a model/comparison stratum; whiskers are sample-bootstrap 95% intervals. " + caption_colour())
            stem = f"F6_tokenization_{metric}_class_{_figure_slug(true_class)}"
            reason = _figure_effect_reason(part_class)
            if reason:
                record(stem, part_class, caption, reason)
                continue
            fig, axes = plt.subplots(2, 1, figsize=(10, 9), layout="constrained")
            extent = float(np.nanmax(np.abs(part_class[["ci_low", "ci_high"]].to_numpy()))) * 1.12
            for ax, platform in zip(axes, FIGURE_PLATFORMS):
                part = part_class[part_class.platform == platform].copy()
                part["_order"] = part.model.map({model: i for i, model in enumerate(FIGURE_MODELS)})
                part = part.sort_values(["_order", "comparison"])
                pos = np.arange(len(part))
                _figure_intervals(ax, part.mean_delta, pos, part.ci_low, part.ci_high, [colours[m] for m in part.model])
                labels = [f"{row.model} | {'DF-aware' if 'deepfake' in row.comparison else 'Manual'}" for row in part.itertuples()]
                ax.set_yticks(pos, labels, fontsize=8)
                ax.invert_yaxis()
                ax.axvline(0, color="#8C96A3", linestyle="--", linewidth=0.9)
                ax.set_xlim(-extent, extent)
                ax.set_xlabel(f"Mean change in {metric.replace('_', ' ')}")
                ax.set_title(_figure_platform(platform))
                _figure_style(ax, "x")
            record(stem, part_class.drop(columns=["_order"], errors="ignore"), caption, fig=fig)


def _draw_error_directions(frame, kind, record, colour, caption_colour):
    prefix = "F3" if kind == "ml_dl" else "F9"
    caption = ("Misclassification counts: Fake-to-Real left of zero and Real-to-Fake right. "
               "The backing table reports each true-class denominator and its conditional error rate. " + caption_colour())
    if frame.empty:
        record(prefix + "_error_directions", frame, caption, "No error-direction evidence rows.")
        return
    key, order = _figure_order(frame, kind)
    limit = max(float(frame.fake_to_real.max()), float(frame.real_to_fake.max()), 1.0) * 1.23
    for platform in FIGURE_PLATFORMS:
        for view in FIGURE_VIEWS:
            part = frame[(frame.platform == platform) & (frame.preprocessing == view)].set_index(key).loc[order].reset_index()
            stem = f"{prefix}_error_directions_{platform}_{view}"
            reason = _figure_support_reason(part, ("fake_support", "real_support"))
            if not reason and (part.fake_to_real.max() == 0 and part.real_to_fake.max() == 0):
                reason = "All error counts are zero; retain the exact table."
            if reason:
                record(stem, part, caption, reason)
                continue
            fig, ax = plt.subplots(figsize=(10, max(4, 0.39 * len(part) + 1.7)), layout="constrained")
            pos, colours = np.arange(len(part)), [colour(row) for _, row in part.iterrows()]
            ax.barh(pos, -part.fake_to_real, color=colours, edgecolor="#3D4A56", linewidth=0.7)
            ax.barh(pos, part.real_to_fake, color=colours, edgecolor="#3D4A56", linewidth=0.7, hatch="//")
            for y, left, right in zip(pos, part.fake_to_real, part.real_to_fake):
                ax.text(-left - limit * 0.016, y, str(int(left)), ha="right", va="center", fontsize=8)
                ax.text(right + limit * 0.016, y, str(int(right)), ha="left", va="center", fontsize=8)
            labels = part.model.tolist() if kind == "transformers" else [_figure_short_label(row).replace("\n", " / ") for _, row in part.iterrows()]
            ax.set_yticks(pos, labels, fontsize=8)
            ax.invert_yaxis()
            ax.axvline(0, color="#8C96A3", linewidth=0.9)
            ax.set_xlim(-limit, limit)
            ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: str(abs(int(value)))))
            ax.set_xlabel("Fake-to-Real count (left); Real-to-Fake count (right, hatched)")
            ax.set_title(f"{_figure_platform(platform)} | {view.replace('_', ' ').title()}")
            _figure_style(ax, "x")
            record(stem, part, caption, fig=fig)


def _draw_transitions(frame, kind, record):
    prefix = "F4" if kind == "ml_dl" else "F7"
    states = ["unchanged_correct", "improved", "degraded", "unchanged_wrong"]
    caption = ("Prediction transitions from original to manual/deepfake-aware preprocessing; each row totals 100% of paired samples. "
               "Colour encodes transition state, not model family or pretraining group: pale to dark = unchanged-correct, "
               "improved, degraded, unchanged-wrong. Narrow segments remain exact in the backing table.")
    if frame.empty:
        record(prefix + "_prediction_transitions", frame, caption, "No paired prediction-transition rows.")
        return
    key, order = _figure_order(frame, kind)
    for platform in FIGURE_PLATFORMS:
        part = frame[frame.platform == platform].copy()
        part["_order"] = part[key].map({value: index for index, value in enumerate(order)})
        part = part.sort_values(["_order", "comparison"]).drop(columns="_order")
        stem = f"{prefix}_prediction_transitions_{platform}"
        reason = _figure_support_reason(part)
        if not reason and not np.allclose(part[states].sum(axis=1), part.n):
            raise ValueError("Transition counts must sum exactly to paired sample support.")
        if not reason and (((part.improved + part.degraded) / part.n) < 0.01).all():
            reason = "Every prediction change rate is below 1%; the exact transition table is more informative."
        if reason:
            record(stem, part, caption, reason)
            continue
        fig, ax = plt.subplots(figsize=(12, max(5.5, 0.32 * len(part) + 2)), layout="constrained")
        pos, left = np.arange(len(part)), np.zeros(len(part))
        for state, state_colour in zip(states, FIGURE_TRANSITION_COLOURS):
            widths = part[state].to_numpy() / part.n.to_numpy() * 100
            ax.barh(pos, widths, left=left, color=state_colour, edgecolor="white", linewidth=0.35,
                    label=state.replace("_", "-").capitalize())
            left += widths
        base = part.model.tolist() if kind == "transformers" else [_figure_short_label(row).replace("\n", " / ") for _, row in part.iterrows()]
        labels = [f"{model} | {'DF-aware' if 'deepfake' in comparison else 'Manual'}" for model, comparison in zip(base, part.comparison)]
        ax.set_yticks(pos, labels, fontsize=7.5)
        ax.invert_yaxis()
        ax.set_xlim(0, 100)
        ax.set_xlabel("Paired samples (%)")
        ax.set_title(_figure_platform(platform), pad=40)
        ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.06), fontsize=8,
                  ncol=4, columnspacing=1.2, handlelength=1.4)
        _figure_style(ax, "x")
        record(stem, part, caption, fig=fig)


def _draw_jaccard(frame, record):
    caption = ("Original-view overlap of misclassified sample-index sets. Colour encodes Jaccard similarity on a fixed 0-1 scale, "
               "not model family or pretraining group. Grey diagonal is omitted; an undefined empty-union comparison is marked NA. "
               "The backing table includes intersections, unions and test-set support.")
    original = frame[frame.preprocessing == "original"].copy() if not frame.empty else frame
    if original.empty:
        record("F8_error_jaccard", original, caption, "No original-view error-overlap evidence.")
        return
    for platform in FIGURE_PLATFORMS:
        part = original[original.platform == platform].copy()
        stem = f"F8_error_jaccard_{platform}"
        reason = _figure_support_reason(part)
        if not reason:
            observed_pairs = set(zip(part.model_a, part.model_b))
            if any((a, b) not in observed_pairs and (b, a) not in observed_pairs
                   for a in FIGURE_MODELS for b in FIGURE_MODELS if a != b):
                reason = "The five-model off-diagonal pair coverage is incomplete."
        if reason:
            record(stem, part, caption, reason)
            continue
        matrix = np.full((5, 5), np.nan)
        for row in part.itertuples():
            i, j = FIGURE_MODELS.index(row.model_a), FIGURE_MODELS.index(row.model_b)
            if i != j and row.union > 0:
                if not 0 <= row.intersection <= row.union or not np.isclose(row.jaccard, row.intersection / row.union):
                    raise ValueError("Jaccard values must equal intersection/union and lie in [0,1].")
                if np.isfinite(matrix[i, j]) and not np.isclose(matrix[i, j], row.jaccard):
                    raise ValueError("Conflicting symmetric Jaccard evidence.")
                matrix[i, j] = matrix[j, i] = row.jaccard
        finite = matrix[np.isfinite(matrix)]
        if len(finite) == 0 or np.ptp(finite) < 1e-12:
            record(stem, part, caption, "All defined pairwise similarities are identical or undefined; retain the exact table.")
            continue
        fig, ax = plt.subplots(figsize=(7.6, 6.5), layout="constrained")
        cmap = plt.get_cmap("Blues").copy()
        cmap.set_bad("#E4E7EB")
        im = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, vmin=0, vmax=1)
        for i in range(5):
            for j in range(5):
                value = matrix[i, j]
                label = "-" if i == j else f"{value:.2f}" if np.isfinite(value) else "NA"
                ax.text(j, i, label, va="center", ha="center", fontsize=10,
                        color="white" if np.isfinite(value) and value > 0.58 else "#26313B")
        ax.set_xticks(range(5), FIGURE_MODELS, rotation=40, ha="right", fontsize=8.5)
        ax.set_yticks(range(5), FIGURE_MODELS, fontsize=8.5)
        ax.set_title(_figure_platform(platform))
        ax.axhline(1.5, color="#D3D8DE", linestyle=":", linewidth=1.0)
        ax.axvline(1.5, color="#D3D8DE", linestyle=":", linewidth=1.0)
        fig.colorbar(im, ax=ax, fraction=0.045, pad=0.025, label="Error-set Jaccard similarity")
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        record(stem, part, caption, fig=fig)


## A.1.1 Configuration and Evidence Inventory

**Purpose:** Verify evidence provenance, coverage, labels, split reconstruction, and view identity.

**Inputs:** Platform split CSVs, source preprocessing code, prediction CSVs, stored metrics, multilingual run metadata.

**Outputs:** 90 platform/configuration/view rows; validated matched sample views and inventory.

**Statistical method:** Exact row/key/label/text checks, not a fitted model. De-leak YouTube before constructing views.

**Table/figure decision:** Configuration and validation tables only. Stop rather than silently omit unmatched evidence.

In [ ]:
predictions,samples,stored_metrics,inventory,data_checks=load_evidence(ROOT,KIND)
checks.extend(data_checks)
assert len(stored_metrics)==90, 'Unexpected result coverage'
tables['configuration_inventory']=inventory
tables['execution_parameters']=pd.DataFrame([{'seed':42,'bootstrap_resamples':N_BOOT,
    'scope':'in_domain','kind':KIND,'expected_results':90,'labels':'0=fake; 1=real'}])
display(inventory.head(15))

## A.1.2 Performance Across Preprocessing

**Purpose:** Recompute metrics for every fixed configuration and view.

**Inputs:** Aligned hard predictions and stored Macro-F1 values.

**Outputs:** Accuracy, macro precision/recall/F1 and sample-bootstrap CIs; source reconciliation.

**Statistical method:** Empirical sample bootstrap via multinomial confusion-cell counts; 95% percentile intervals.

**Table/figure decision:** F1 reproduces the six-panel style: columns=platform, rows=view, family-shaded bars and capped CIs. Focused y limits are explicit; exact tables remain authoritative.

In [ ]:
performance=performance_evidence(predictions,stored_metrics,N_BOOT)
tables['performance']=performance
checks.append({'check':'macro_f1_reproduction_1e-10','passed':bool(performance.reconciliation_abs_error.max()<1e-10),
    'detail':str(performance.reconciliation_abs_error.max())})
display(performance[['platform','configuration','preprocessing','macro_f1','ci_low','ci_high']].head(15))

## A.1.3 Mathematical Effects of Preprocessing

**Purpose:** Measure what each platform-specific normalization removes, retains, or amplifies.

**Inputs:** Original/manual/deepfake-aware text for the same test sample.

**Outputs:** 33-feature dictionary, per-sample features, paired changes and BH-FDR families.

**Statistical method:** Paired medians, increase/decrease/unchanged %, Wilcoxon, rank-biserial, bootstrap CI; secondary IQR shift.

**Table/figure decision:** Exact feature-change tables by platform and true class; feature profiles are linked to prediction transitions later.

In [ ]:
sample_features,feature_changes,feature_definitions=preprocessing_patterns(samples,N_BOOT)
tables['sample_features']=sample_features
tables['feature_changes']=feature_changes
tables['feature_dictionary']=feature_definitions
display(feature_changes.head(12))

## A.1.4 Prediction Improvements and Degradations

**Purpose:** Connect preprocessing to corrected and newly introduced errors without causal claims.

**Inputs:** Same-configuration original/view predictions and paired mathematical features.

**Outputs:** Macro-F1 deltas, exact McNemar, four-state counts, class-specific changes and descriptive feature profiles.

**Statistical method:** Paired bootstrap for Macro-F1; exact binomial McNemar. Feature profiles retain one sample per configuration/state.

**Table/figure decision:** F2 delta forest for A.1 and F4/F7 transitions. All state denominators and low-support flags remain in tables.

In [ ]:
prediction_changes,class_changes,prediction_pairs=paired_predictions(predictions,N_BOOT)
tables['prediction_changes']=prediction_changes
tables['prediction_changes_by_class']=class_changes
tables['prediction_transitions']=prediction_pairs
tables['transition_feature_profiles']=transition_feature_profiles(sample_features,prediction_pairs)
assert (prediction_changes[STATES].sum(axis=1)==prediction_changes.n).all()
checks.append({'check':'four_transition_denominators','passed':True,'detail':'States sum to every matched run support'})
display(prediction_changes[['platform','configuration','comparison','macro_f1_delta','ci_low','ci_high','mcnemar_p']].head(12))

## A.1.5 FP/FN Directions

**Purpose:** Separate missed machine-generated text from human-text false alarms.

**Inputs:** All aligned predictions with verified 0=fake and 1=real semantics.

**Outputs:** Exact directional counts, class supports, class-conditional rates.

**Statistical method:** Fake→Real / fake support; Real→Fake / real support. No ambiguous numeric-positive-label convention.

**Table/figure decision:** F3/F9 diverging counts by platform/view with direct count labels; rates in authoritative tables.

In [ ]:
tables['error_directions']=direction_evidence(predictions)
display(tables['error_directions'].head(10))

## A.1.6 Errors by Deception Type, Dialect, and Topic

**Purpose:** Locate descriptive error concentrations while retaining every category denominator.

**Inputs:** Canonical metadata joined by platform and sample_idx; fake-only deception subtypes.

**Outputs:** Error counts, supports and rates by platform, true class, model and view.

**Statistical method:** Exact stratified proportions, not pooled platform results. Missing categories remain visible.

**Table/figure decision:** Tables only; support below 30 is flagged, not silently removed.

In [ ]:
tables.update(metadata_evidence(predictions))
checks.append({'check':'metadata_category_denominators','passed':True,'detail':'Category supports reconcile to each run/class; missing values retained'})
display(tables['errors_by_deception_type'].head(12))

## A.1.7 Cross-Configuration Error Agreement

**Purpose:** Identify common errors without treating repeated model predictions as independent samples.

**Inputs:** Every configuration evaluated on identical sample IDs.

**Outputs:** Consensus votes, hard-sample support, pairwise intersections/unions and Jaccard.

**Statistical method:** Set intersection/union; empty-union similarity is undefined and explicitly flagged. No pooled inference.

**Table/figure decision:** Consensus tables; A.2 additionally uses original-view 5×5 F8 heatmaps with fixed 0–1 scale.

In [ ]:
consensus,error_jaccard=agreement_evidence(predictions)
tables['consensus_errors']=consensus
tables['error_jaccard']=error_jaccard
display(consensus.head(10))

## A.1.8 Representative Misclassifications

**Purpose:** Provide reproducible inspection examples while exporting all errors.

**Inputs:** Complete misclassifications, consensus votes and improved/degraded transitions.

**Outputs:** Complete error export plus consensus-hard and preprocessing-sensitive examples with selection rules.

**Statistical method:** Deterministic selection by error votes then sample_idx, or first sample_idx per transition; no random cherry-picking.

**Table/figure decision:** Tables only. Examples illustrate measured associations; they do not establish linguistic causes.

In [ ]:
all_errors,hard_examples,sensitive_examples=example_evidence(predictions,prediction_pairs,consensus)
tables['complete_misclassifications']=all_errors
tables['representative_hard_examples']=hard_examples
tables['representative_sensitive_examples']=sensitive_examples
assert len(all_errors)==int(predictions.y_true.ne(predictions.y_pred).sum())
checks.append({'check':'complete_error_export','passed':True,'detail':f'{len(all_errors)} misclassified prediction rows; none sampled out'})
display(hard_examples[['platform','sample_idx','preprocessing','error_votes','configuration_count']].head(12))

## Exported Evidence

**Purpose:** Export reproducible, separate notebook evidence and apply figure gates.

**Inputs:** All validated tables; figure specification and stated palette meaning.

**Outputs:** One workbook, UTF-8-SIG CSVs, table/figure/validation manifests, 600-DPI PNGs and same-stem vector PDFs.

**Statistical method:** No additional hypothesis/model selection. Suppression affects presentation only, never table export.

**Table/figure decision:** Every emitted/suppressed figure has a backing CSV and a recorded caption/reason; display compact previews.

In [ ]:
figure_manifest=generate_figures(KIND,tables,OUTPUT_DIR)
checks.extend([
 {'check':'evaluation_only','passed':True,'detail':'No detector fitting, tuning, threshold selection, or target-test model selection in this notebook'},
 {'check':'single_seed_limit','passed':True,'detail':'One seed; bootstrap represents test-sample uncertainty, not training variance'},
 {'check':'platform_generator_confounding','passed':True,'detail':'Platform and generator effects cannot be isolated'},
])
assert all(bool(x['passed']) for x in checks), 'A required validation check failed'
manifest,workbook_path=export_evidence(tables,figure_manifest,checks,OUTPUT_DIR)
display(manifest[['table','rows','excel_sheet']])
print(f'Saved workbook: {workbook_path}')
from PIL import Image as PILImage
from io import BytesIO
for row in figure_manifest.to_dict('records'):
    if row.get('status') not in ['generated','emitted','exported']: continue
    path=Path(row['png'])
    if not path.is_absolute():path=OUTPUT_DIR/path
    with PILImage.open(path) as image:
        image.thumbnail((1600,1600));buffer=BytesIO();image.save(buffer,format='PNG')
    display(Image(data=buffer.getvalue(),width=900))
    display(Markdown(row['caption']))

## Interpretation and Limitations

- One seed does not estimate training variance; bootstrap CIs describe test-sample uncertainty.
- Platform and generator effects remain confounded. No causal claim or universal platform invariance follows.
- No source-pair ID exists. Exact de-leakage does not establish independence of human/synthetic source pairs.
- Historical platform-specific preprocessing differs deliberately. These are the implemented pipelines, not a universal recipe.
- Mathematical indicators are operational proxies. Compression includes codec overhead; script ranges are not dialect detectors.
- McNemar concerns correctness, not the magnitude or cause of a Macro-F1 change. Small p-values do not imply practical importance.
- The new analyses neither retrain detectors nor select configurations on these test results. Findings are associative.
- Figure gates are presentation rules; exact tables include nonsignificant, zero, missing-category and low-support results.
- Read each caption: A.1 colour means family; A.2 colour means pretraining group. Transition/heatmap colours encode states/similarity.

## Takeaways

**Purpose:** Summarize only executed evidence.

**Inputs:** Validated results and generated manifests.

**Outputs:** Observed coverage, delta ranges and output links.

**Statistical method:** Descriptive summary; no causal interpretation.

**Table/figure decision:** Text summary only; exact values remain in the workbook.

In [ ]:
summary=[]
for platform,g in performance.groupby('platform'):
    delta=prediction_changes[prediction_changes.platform.eq(platform)]
    summary.append(f"{platform}: {len(g)} result rows; preprocessing Macro-F1 deltas range "
                   f"from {100*delta.macro_f1_delta.min():+.2f} to {100*delta.macro_f1_delta.max():+.2f} percentage points.")
display(Markdown('\n\n'.join(summary)+f'\n\nComplete misclassified prediction rows exported: **{len(all_errors):,}**. '
    'These are prediction rows across configurations/views, not unique texts.'))